# 03 Machine Learning + Deep Learning

Merged notebook for modules 03a-04f in tutorial order.

In [ ]:
# @title ⚙️ Environment Setup (Run this first!)
!pip install pymatgen numpy matplotlib scikit-learn torch -q
print("✅ Packages installed")

In [ ]:
# @title 📂 Load Tutorial Data
import os

REPO = "MRS_CH08_Tutorial"
REPO_URL = "https://github.com/Szymanski-Group/MRS_CH08_Tutorial.git"

if os.path.basename(os.getcwd()) == REPO:
    print("✅ Data already present")
elif os.path.exists(REPO):
    os.chdir(REPO)
    print("✅ Data already present")
else:
    !git clone {REPO_URL} -q
    os.chdir(REPO)
    print("✅ Data loaded successfully")


## 03a - Model Training Validation

In [ ]:
from IPython.display import Image, display

# Inline tutorial script
from pathlib import Path
import csv

# For handling arrays
import numpy as np

# For plotting
import matplotlib.pyplot as plt

# To load structures and compute XRD stick patterns
from pymatgen.core import Structure
from pymatgen.analysis.diffraction.xrd import XRDCalculator

# Conventional ML models
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score


# Input/output
EXPERIMENT_DIR = Path("data/exp_patterns/one_phase")
REFERENCE_DIR = Path("data/reference_structures")
OUTPUT_DIR = Path("outputs/ml/conv")

# Pattern settings
MIN_ANGLE = 10.0
MAX_ANGLE = 80.0
NUM_POINTS = 1800
WAVELENGTH = "CuKa"
WAVELENGTH_ANGSTROM = 1.5406
REFERENCE_INTENSITY_THRESHOLD = 1.0

# Synthetic-data settings
SYNTH_SAMPLES_PER_FORMULA = 80
RANDOM_SEED = 42

# Artifact ranges (inspired by Slide-34 all-artifacts)
UNIFORM_SHIFT_RANGE = (-0.15, 0.15)
SAMPLE_DISPLACEMENT_RANGE_MM = (-0.20, 0.20)
GONIOMETER_RADIUS_MM = 240.0

U_RANGE = (0.01, 0.06)
V_RANGE = (-0.02, 0.01)
W_RANGE = (0.002, 0.010)
SIZE_NM_RANGE = (8.0, 80.0)
MICROSTRAIN_RANGE = (0.0, 0.003)

FWHM_RANGE = (0.08, 0.45)
ETA_RANGE = (0.10, 0.70)

BACKGROUND_SCALE_RANGE = (0.05, 0.30)
HUMP_SCALE_RANGE = (0.02, 0.20)
NOISE_SCALE_RANGE = (0.002, 0.020)

# Split and reporting
VAL_FRACTION = 0.20


def normalize_0_100(y):
    y = np.asarray(y, dtype=float)
    y = y - y.min()
    return 100.0 * y / np.clip(y.max(), 1e-12, None)


def sample_displacement_shift(two_theta_deg, displacement_mm):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    d_relative_change = displacement_mm / GONIOMETER_RADIUS_MM * np.cos(theta_rad) ** 2
    return np.rad2deg(-d_relative_change * np.tan(theta_rad))


def instrumental_fwhm(two_theta_deg, u, v, w):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    tan_theta = np.tan(theta_rad)
    fwhm_sq = u * tan_theta**2 + v * tan_theta + w
    return np.sqrt(np.clip(fwhm_sq, 1e-4, None))


def size_fwhm(two_theta_deg, size_nm):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    wavelength_nm = WAVELENGTH_ANGSTROM / 10.0
    beta_rad = 0.9 * wavelength_nm / (size_nm * np.cos(theta_rad))
    return np.rad2deg(beta_rad)


def strain_fwhm(two_theta_deg, microstrain):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    beta_rad = 4.0 * microstrain * np.tan(theta_rad)
    return np.rad2deg(beta_rad)


def pseudo_voigt_profile(two_theta_grid, centers, fwhm, eta):
    dx = two_theta_grid[:, None] - centers[None, :]
    sigma = np.clip(fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0))), 1e-6, None)
    gamma = np.clip(fwhm / 2.0, 1e-6, None)
    gauss = np.exp(-0.5 * (dx / sigma[None, :]) ** 2)
    lorentz = (gamma[None, :] ** 2) / (dx**2 + gamma[None, :] ** 2)
    return (1.0 - eta) * gauss + eta * lorentz


def load_reference_sticks(cif_files):
    calc = XRDCalculator(wavelength=WAVELENGTH)
    refs = []

    for cif_file in cif_files:
        pattern = calc.get_pattern(Structure.from_file(cif_file), two_theta_range=(MIN_ANGLE, MAX_ANGLE))
        peak_pos = np.asarray(pattern.x, dtype=float)
        peak_int = np.asarray(pattern.y, dtype=float)

        keep = peak_int >= REFERENCE_INTENSITY_THRESHOLD
        peak_pos = peak_pos[keep]
        peak_int = normalize_0_100(peak_int[keep])

        formula = cif_file.stem.split("_", 1)[0]
        refs.append({"phase": cif_file.stem, "formula": formula, "peak_pos": peak_pos, "peak_int": peak_int})

    return refs


def simulate_artifact_profile(two_theta_grid, base_pos, base_int, rng):
    if len(base_pos) == 0:
        return np.zeros_like(two_theta_grid)

    # Intensity perturbation (texture-like random reweighting)
    peak_int = base_int * np.exp(rng.normal(0.0, 0.25, size=len(base_int)))

    # Peak-position perturbations
    uniform_shift = rng.uniform(*UNIFORM_SHIFT_RANGE)
    displacement = rng.uniform(*SAMPLE_DISPLACEMENT_RANGE_MM)
    peak_pos = base_pos + uniform_shift + sample_displacement_shift(base_pos, displacement)

    keep = (peak_pos >= MIN_ANGLE - 1.0) & (peak_pos <= MAX_ANGLE + 1.0)
    peak_pos = peak_pos[keep]
    peak_int = peak_int[keep]
    if len(peak_pos) == 0:
        return np.zeros_like(two_theta_grid)

    # Peak broadening model: instrument + size + strain + random extra width
    u = rng.uniform(*U_RANGE)
    v = rng.uniform(*V_RANGE)
    w = rng.uniform(*W_RANGE)
    size_nm = rng.uniform(*SIZE_NM_RANGE)
    microstrain = rng.uniform(*MICROSTRAIN_RANGE)

    fwhm = np.sqrt(
        instrumental_fwhm(peak_pos, u, v, w) ** 2
        + size_fwhm(peak_pos, size_nm) ** 2
        + strain_fwhm(peak_pos, microstrain) ** 2
    )
    fwhm += rng.uniform(*FWHM_RANGE)
    eta = rng.uniform(*ETA_RANGE)

    peaks = pseudo_voigt_profile(two_theta_grid, peak_pos, fwhm, eta) @ peak_int
    peaks = normalize_0_100(peaks)

    # Smooth polynomial background
    x_cheb = 2.0 * (two_theta_grid - MIN_ANGLE) / (MAX_ANGLE - MIN_ANGLE) - 1.0
    coeffs = np.array(
        [
            1.0,
            rng.uniform(-0.5, 0.5),
            rng.uniform(-0.4, 0.4),
            rng.uniform(-0.2, 0.2),
            rng.uniform(-0.1, 0.1),
        ]
    )
    background = np.polynomial.chebyshev.chebval(x_cheb, coeffs)
    background -= background.min()
    background /= np.clip(background.max(), 1e-12, None)
    background *= rng.uniform(*BACKGROUND_SCALE_RANGE) * peaks.max()

    # Amorphous hump + Gaussian noise
    center = rng.uniform(18.0, 35.0)
    width = rng.uniform(5.0, 12.0)
    hump = rng.uniform(*HUMP_SCALE_RANGE) * peaks.max() * np.exp(-0.5 * ((two_theta_grid - center) / width) ** 2)

    noise_sigma = rng.uniform(*NOISE_SCALE_RANGE) * peaks.max()
    noise = rng.normal(0.0, noise_sigma, size=len(two_theta_grid))

    y = peaks + background + hump + noise
    y -= y.min()
    return normalize_0_100(y)


def build_synthetic_dataset(reference_sticks, two_theta_grid, rng):
    # Group reference entries by formula so classes remain balanced.
    grouped = {}
    for ref in reference_sticks:
        grouped.setdefault(ref["formula"], []).append(ref)

    X = []
    y = []

    for formula, entries in sorted(grouped.items()):
        for _ in range(SYNTH_SAMPLES_PER_FORMULA):
            ref = entries[rng.integers(0, len(entries))]
            profile = simulate_artifact_profile(two_theta_grid, ref["peak_pos"], ref["peak_int"], rng)
            X.append(profile)
            y.append(formula)

    return np.asarray(X), np.asarray(y)


def preprocess_experimental_pattern(xy_file, two_theta_grid):
    data = np.loadtxt(xy_file)
    x = data[:, 0]
    y = data[:, 1]

    keep = (x >= MIN_ANGLE) & (x <= MAX_ANGLE)
    x = x[keep]
    y = y[keep]

    y_interp = np.interp(two_theta_grid, x, y)
    y_interp = np.clip(y_interp - np.percentile(y_interp, 5.0), 0.0, None)
    y_interp = normalize_0_100(y_interp)
    return y_interp


def tune_model(X_train, y_train, X_val, y_val, model_builder, param_grid):
    best = None
    for params in param_grid:
        model = model_builder(**params)
        model.fit(X_train, y_train)
        val_acc = accuracy_score(y_val, model.predict(X_val))
        if best is None or val_acc > best["val_acc"]:
            best = {"model": model, "val_acc": val_acc, "params": params}
    return best


def plot_accuracy_summary(model_results):
    model_names = list(model_results.keys())
    val_acc = [model_results[m]["val_acc"] for m in model_names]
    test_acc = [model_results[m]["test_acc"] for m in model_names]

    x = np.arange(len(model_names))
    width = 0.36

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar(
        x - width / 2,
        val_acc,
        width=width,
        color="tab:blue",
        edgecolor="black",
        linewidth=1.0,
        label="Validation",
    )
    ax.bar(
        x + width / 2,
        test_acc,
        width=width,
        color="tab:red",
        edgecolor="black",
        linewidth=1.0,
        label="Test",
    )

    ax.set_xticks(x)
    ax.set_xticklabels(model_names, fontsize=16)
    ax.tick_params(axis="y", labelsize=16)
    ax.set_ylim(0.0, 1.05)
    ax.set_ylabel("Accuracy", fontsize=18, labelpad=12)
    ax.legend(fontsize=18, loc='lower right', framealpha=1)
    ax.grid(axis="y", alpha=0.25)

    out_file = OUTPUT_DIR / "model_accuracy_summary.png"
    plt.tight_layout()
    plt.savefig(out_file, dpi=200)
    plt.close(fig)
    print(f"Saved plot: {out_file}")


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(RANDOM_SEED)
    two_theta_grid = np.linspace(MIN_ANGLE, MAX_ANGLE, NUM_POINTS)

    reference_sticks = load_reference_sticks(sorted(REFERENCE_DIR.glob("*.cif")))
    X, y = build_synthetic_dataset(reference_sticks, two_theta_grid, rng)

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=VAL_FRACTION,
        random_state=RANDOM_SEED,
        stratify=y,
    )

    exp_files = sorted(EXPERIMENT_DIR.glob("*.xy"))
    X_exp = np.asarray([preprocess_experimental_pattern(f, two_theta_grid) for f in exp_files])
    y_exp = np.asarray([f.stem for f in exp_files])

    print("\n=== Conventional ML Phase-ID Demo (k-NN, Random Forest, SVM) ===")
    print(f"Synthetic training+validation samples: {len(X)}")
    print(f"Number of phase classes: {len(np.unique(y))}")
    print(f"Training samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    print(f"Test patterns (experimental): {len(X_exp)}")

    model_results = {
        "k-NN": tune_model(
            X_train,
            y_train,
            X_val,
            y_val,
            model_builder=lambda n_neighbors: make_pipeline(
                StandardScaler(),
                KNeighborsClassifier(n_neighbors=n_neighbors, weights="distance"),
            ),
            param_grid=[{"n_neighbors": k} for k in [1, 3, 5, 7, 11]],
        ),
        "Random Forest": tune_model(
            X_train,
            y_train,
            X_val,
            y_val,
            model_builder=lambda n_estimators, max_depth, min_samples_leaf: RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                random_state=RANDOM_SEED,
                n_jobs=-1,
            ),
            param_grid=[
                {"n_estimators": 300, "max_depth": d, "min_samples_leaf": l}
                for d in [None, 15, 30]
                for l in [1, 2, 4]
            ],
        ),
        "SVM": tune_model(
            X_train,
            y_train,
            X_val,
            y_val,
            model_builder=lambda C, gamma: make_pipeline(
                StandardScaler(),
                SVC(kernel="rbf", C=C, gamma=gamma, probability=False),
            ),
            param_grid=[{"C": c, "gamma": g} for c in [1.0, 5.0, 10.0] for g in ["scale", 0.01, 0.001]],
        ),
    }

    prediction_rows = []

    for model_name, result in model_results.items():
        model = result["model"]
        y_pred_exp = model.predict(X_exp)
        test_acc = accuracy_score(y_exp, y_pred_exp)
        result["test_acc"] = test_acc

        print(f"\n{model_name}")
        print(f"  Best params: {result['params']}")
        print(f"  Validation accuracy: {result['val_acc']:.3f}")
        print(f"  Test accuracy: {test_acc:.3f}")

        print("  Test predictions:")
        for true_label, pred_label in zip(y_exp, y_pred_exp):
            correct = (true_label == pred_label)
            mark = "✓" if correct else "x"
            print(f"    {true_label:8s} -> {pred_label:8s} {mark}")
            prediction_rows.append(
                {
                    "model": model_name,
                    "pattern": true_label,
                    "true_phase": true_label,
                    "predicted_phase": pred_label,
                    "correct": int(correct),
                }
            )

    plot_accuracy_summary(model_results)

    metrics_file = OUTPUT_DIR / "model_metrics.csv"
    with open(metrics_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["model", "validation_accuracy", "test_accuracy", "best_params"],
        )
        writer.writeheader()
        for name, result in model_results.items():
            writer.writerow(
                {
                    "model": name,
                    "validation_accuracy": result["val_acc"],
                    "test_accuracy": result["test_acc"],
                    "best_params": str(result["params"]),
                }
            )

    pred_file = OUTPUT_DIR / "experimental_predictions.csv"
    with open(pred_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["model", "pattern", "true_phase", "predicted_phase", "correct"],
        )
        writer.writeheader()
        writer.writerows(prediction_rows)

    print(f"\nSaved metrics: {metrics_file}")
    print(f"Saved predictions: {pred_file}")


# 03a — Conventional ML: Training & Validation

We train k-NN, Random Forest, and SVM classifiers on synthetic single-phase profiles and test on experimental data.

## Runtime Note
For a live tutorial, we use a smaller synthetic dataset. Increase `SYNTH_SAMPLES_PER_FORMULA` for higher-fidelity training offline.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
SYNTH_SAMPLES_PER_FORMULA = 12  # Reduced for Colab demo speed
main()


## What To Observe
Compare validation vs experimental-test accuracy across the three model families.

In [ ]:
display(Image("outputs/ml/conv/model_accuracy_summary.png"))

## Summary
- Synthetic augmentation enables supervised learning with limited experimental labels.
- Different model families show different generalization behavior.
- Validation and test gaps highlight domain-shift effects.

## Next Steps
Continue to the next section below in this notebook.

## 03b - Multiphase ML

In [ ]:
from IPython.display import Image, display

# Inline tutorial script
from pathlib import Path
import csv

# For handling arrays
import numpy as np

# For plotting
import matplotlib.pyplot as plt

# To load structures and compute XRD stick patterns
from pymatgen.core import Structure
from pymatgen.analysis.diffraction.xrd import XRDCalculator

# Conventional ML models
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import precision_score, recall_score, f1_score


# Input/output
EXPERIMENT_DIR = Path("data/exp_patterns/multi_phase")
REFERENCE_DIR = Path("data/reference_structures")
OUTPUT_DIR = Path("outputs/ml/multiphase")

# Pattern settings
MIN_ANGLE = 10.0
MAX_ANGLE = 80.0
NUM_POINTS = 1400
WAVELENGTH = "CuKa"
WAVELENGTH_ANGSTROM = 1.5406
REFERENCE_INTENSITY_THRESHOLD = 1.0

# Synthetic-data settings
SYNTH_SAMPLES_PER_FORMULA = 60
RANDOM_SEED = 42
SINGLE_PHASE_FRACTION = 0.15

# Artifact ranges (same style as Slide-34)
UNIFORM_SHIFT_RANGE = (-0.12, 0.12)
SAMPLE_DISPLACEMENT_RANGE_MM = (-0.18, 0.18)
GONIOMETER_RADIUS_MM = 240.0

U_RANGE = (0.01, 0.06)
V_RANGE = (-0.02, 0.01)
W_RANGE = (0.002, 0.010)
SIZE_NM_RANGE = (8.0, 80.0)
MICROSTRAIN_RANGE = (0.0, 0.003)

FWHM_RANGE = (0.08, 0.45)
ETA_RANGE = (0.10, 0.70)

BACKGROUND_SCALE_RANGE = (0.05, 0.28)
HUMP_SCALE_RANGE = (0.02, 0.20)
NOISE_SCALE_RANGE = (0.002, 0.020)

# Split
VAL_FRACTION = 0.20


def normalize_0_100(y):
    y = np.asarray(y, dtype=float)
    y = y - y.min()
    return 100.0 * y / np.clip(y.max(), 1e-12, None)


def sample_displacement_shift(two_theta_deg, displacement_mm):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    d_relative_change = displacement_mm / GONIOMETER_RADIUS_MM * np.cos(theta_rad) ** 2
    return np.rad2deg(-d_relative_change * np.tan(theta_rad))


def instrumental_fwhm(two_theta_deg, u, v, w):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    tan_theta = np.tan(theta_rad)
    fwhm_sq = u * tan_theta**2 + v * tan_theta + w
    return np.sqrt(np.clip(fwhm_sq, 1e-4, None))


def size_fwhm(two_theta_deg, size_nm):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    wavelength_nm = WAVELENGTH_ANGSTROM / 10.0
    beta_rad = 0.9 * wavelength_nm / (size_nm * np.cos(theta_rad))
    return np.rad2deg(beta_rad)


def strain_fwhm(two_theta_deg, microstrain):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    beta_rad = 4.0 * microstrain * np.tan(theta_rad)
    return np.rad2deg(beta_rad)


def pseudo_voigt_profile(two_theta_grid, centers, fwhm, eta):
    dx = two_theta_grid[:, None] - centers[None, :]
    sigma = np.clip(fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0))), 1e-6, None)
    gamma = np.clip(fwhm / 2.0, 1e-6, None)
    gauss = np.exp(-0.5 * (dx / sigma[None, :]) ** 2)
    lorentz = (gamma[None, :] ** 2) / (dx**2 + gamma[None, :] ** 2)
    return (1.0 - eta) * gauss + eta * lorentz


def load_reference_sticks(cif_files):
    calc = XRDCalculator(wavelength=WAVELENGTH)
    refs_by_formula = {}

    for cif_file in cif_files:
        pattern = calc.get_pattern(Structure.from_file(cif_file), two_theta_range=(MIN_ANGLE, MAX_ANGLE))
        peak_pos = np.asarray(pattern.x, dtype=float)
        peak_int = np.asarray(pattern.y, dtype=float)

        keep = peak_int >= REFERENCE_INTENSITY_THRESHOLD
        peak_pos = peak_pos[keep]
        peak_int = normalize_0_100(peak_int[keep])

        formula = cif_file.stem.split("_", 1)[0]
        refs_by_formula.setdefault(formula, []).append({"phase": cif_file.stem, "peak_pos": peak_pos, "peak_int": peak_int})

    return refs_by_formula


def simulate_component_profile(two_theta_grid, base_pos, base_int, rng):
    if len(base_pos) == 0:
        return np.zeros_like(two_theta_grid)

    # Intensity perturbation (texture-like random reweighting)
    peak_int = base_int * np.exp(rng.normal(0.0, 0.25, size=len(base_int)))

    # Peak-position perturbations
    uniform_shift = rng.uniform(*UNIFORM_SHIFT_RANGE)
    displacement = rng.uniform(*SAMPLE_DISPLACEMENT_RANGE_MM)
    peak_pos = base_pos + uniform_shift + sample_displacement_shift(base_pos, displacement)

    keep = (peak_pos >= MIN_ANGLE - 1.0) & (peak_pos <= MAX_ANGLE + 1.0)
    peak_pos = peak_pos[keep]
    peak_int = peak_int[keep]
    if len(peak_pos) == 0:
        return np.zeros_like(two_theta_grid)

    # Peak broadening model
    u = rng.uniform(*U_RANGE)
    v = rng.uniform(*V_RANGE)
    w = rng.uniform(*W_RANGE)
    size_nm = rng.uniform(*SIZE_NM_RANGE)
    microstrain = rng.uniform(*MICROSTRAIN_RANGE)

    fwhm = np.sqrt(
        instrumental_fwhm(peak_pos, u, v, w) ** 2
        + size_fwhm(peak_pos, size_nm) ** 2
        + strain_fwhm(peak_pos, microstrain) ** 2
    )
    fwhm += rng.uniform(*FWHM_RANGE)
    eta = rng.uniform(*ETA_RANGE)

    profile = pseudo_voigt_profile(two_theta_grid, peak_pos, fwhm, eta) @ peak_int
    return profile / np.clip(profile.max(), 1e-12, None)


def simulate_multiphase_profile(two_theta_grid, formula_list, refs_by_formula, rng):
    components = []

    for formula in formula_list:
        ref = refs_by_formula[formula][rng.integers(0, len(refs_by_formula[formula]))]
        comp = simulate_component_profile(two_theta_grid, ref["peak_pos"], ref["peak_int"], rng)
        components.append(comp)

    # Random phase fractions (sum to 1)
    weights = rng.dirichlet(np.ones(len(components)) * 1.5)
    peaks = np.zeros_like(two_theta_grid)
    for w, comp in zip(weights, components):
        peaks += w * comp
    peaks = normalize_0_100(peaks)

    # Global background
    x_cheb = 2.0 * (two_theta_grid - MIN_ANGLE) / (MAX_ANGLE - MIN_ANGLE) - 1.0
    coeffs = np.array([1.0, rng.uniform(-0.5, 0.5), rng.uniform(-0.4, 0.4), rng.uniform(-0.2, 0.2), rng.uniform(-0.1, 0.1)])
    background = np.polynomial.chebyshev.chebval(x_cheb, coeffs)
    background -= background.min()
    background /= np.clip(background.max(), 1e-12, None)
    background *= rng.uniform(*BACKGROUND_SCALE_RANGE) * peaks.max()

    # Amorphous hump + Gaussian noise
    center = rng.uniform(18.0, 35.0)
    width = rng.uniform(5.0, 12.0)
    hump = rng.uniform(*HUMP_SCALE_RANGE) * peaks.max() * np.exp(-0.5 * ((two_theta_grid - center) / width) ** 2)

    noise_sigma = rng.uniform(*NOISE_SCALE_RANGE) * peaks.max()
    noise = rng.normal(0.0, noise_sigma, size=len(two_theta_grid))

    y = peaks + background + hump + noise
    y -= y.min()
    return normalize_0_100(y)


def build_synthetic_multiphase_dataset(refs_by_formula, two_theta_grid, rng):
    formulas = sorted(refs_by_formula.keys())
    X = []
    y_labels = []

    # Anchor each formula so every phase appears many times in training data.
    for anchor in formulas:
        others = [f for f in formulas if f != anchor]
        for _ in range(SYNTH_SAMPLES_PER_FORMULA):
            # Include a small fraction of pure (1-phase) samples.
            # Keep the original 2-phase vs 3-phase ratio for the remaining fraction.
            n_components = int(
                rng.choice(
                    [1, 2, 3],
                    p=[
                        SINGLE_PHASE_FRACTION,
                        (1.0 - SINGLE_PHASE_FRACTION) * 0.65,
                        (1.0 - SINGLE_PHASE_FRACTION) * 0.35,
                    ],
                )
            )
            chosen_others = list(rng.choice(others, size=n_components - 1, replace=False))
            labels = sorted([anchor] + chosen_others)

            profile = simulate_multiphase_profile(two_theta_grid, labels, refs_by_formula, rng)
            X.append(profile)
            y_labels.append(labels)

    return np.asarray(X), y_labels


def preprocess_experimental_pattern(xy_file, two_theta_grid):
    data = np.loadtxt(xy_file)
    x = data[:, 0]
    y = data[:, 1]

    keep = (x >= MIN_ANGLE) & (x <= MAX_ANGLE)
    x = x[keep]
    y = y[keep]

    y_interp = np.interp(two_theta_grid, x, y)
    y_interp = np.clip(y_interp - np.percentile(y_interp, 5.0), 0.0, None)
    return normalize_0_100(y_interp)


def get_label_scores(model, X):
    if hasattr(model, "predict_proba"):
        scores = model.predict_proba(X)
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X)
    else:
        scores = model.predict(X)

    if isinstance(scores, list):
        # Some wrappers can return list of per-label arrays.
        cols = []
        for s in scores:
            s = np.asarray(s)
            if s.ndim == 2 and s.shape[1] == 2:
                cols.append(s[:, 1])
            else:
                cols.append(s.ravel())
        return np.column_stack(cols)

    scores = np.asarray(scores)
    if scores.ndim == 1:
        scores = scores[:, None]
    return scores


def predict_with_threshold(scores, threshold):
    y_pred = (scores >= threshold).astype(int)

    # Keep at least one phase prediction per pattern.
    empty = np.where(y_pred.sum(axis=1) == 0)[0]
    if len(empty) > 0:
        best_idx = np.argmax(scores[empty], axis=1)
        y_pred[empty, best_idx] = 1

    return y_pred


def threshold_metrics(y_true_bin, scores, threshold):
    y_pred_bin = predict_with_threshold(scores, threshold)
    precision_micro = precision_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    recall_micro = recall_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    f1_micro = f1_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    return precision_micro, recall_micro, f1_micro, y_pred_bin


def pick_best_threshold(y_true_bin, scores):
    best_t = 0.5
    best_f1 = -1.0
    for t in np.linspace(0.10, 0.90, 41):
        _, _, f1_micro, _ = threshold_metrics(y_true_bin, scores, t)
        if f1_micro > best_f1:
            best_f1 = f1_micro
            best_t = float(t)
    return best_t, best_f1


def tune_model(X_train, y_train, X_val, y_val, model_builder, param_grid):
    best = None
    for params in param_grid:
        model = model_builder(**params)
        model.fit(X_train, y_train)

        val_scores = get_label_scores(model, X_val)
        threshold, f1_micro = pick_best_threshold(y_val, val_scores)

        if best is None or f1_micro > best["val_f1_micro"]:
            best = {
                "model": model,
                "threshold": threshold,
                "val_f1_micro": f1_micro,
                "params": params,
            }
    return best


def labels_from_filename(file_stem):
    # Example: Li2MnO3_MnO_TiO2 -> [Li2MnO3, MnO, TiO2]
    return file_stem.split("_")


def plot_test_metric_summary(model_results):
    model_names = list(model_results.keys())
    test_precision = [model_results[m]["test_precision_micro"] for m in model_names]
    test_recall = [model_results[m]["test_recall_micro"] for m in model_names]
    test_f1 = [model_results[m]["test_f1_micro"] for m in model_names]

    x = np.arange(len(model_names))
    width = 0.24

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar(x - width, test_precision, width=width, color="tab:blue", edgecolor="black", linewidth=1.0, label="Precision")
    ax.bar(x, test_recall, width=width, color="tab:green", edgecolor="black", linewidth=1.0, label="Recall")
    ax.bar(x + width, test_f1, width=width, color="tab:red", edgecolor="black", linewidth=1.0, label="F1-score")

    ax.set_xticks(x)
    ax.set_xticklabels(model_names, fontsize=16)
    ax.tick_params(axis="y", labelsize=16)
    ax.set_ylim(0.0, 1.05)
    ax.set_ylabel("Score", fontsize=18, labelpad=12)
    ax.legend(fontsize=18, loc="lower right", framealpha=1)
    ax.grid(axis="y", alpha=0.25)

    out_file = OUTPUT_DIR / "model_test-metric_summary.png"
    plt.tight_layout()
    plt.savefig(out_file, dpi=200)
    plt.close(fig)
    print(f"Saved plot: {out_file}")


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(RANDOM_SEED)
    two_theta_grid = np.linspace(MIN_ANGLE, MAX_ANGLE, NUM_POINTS)

    refs_by_formula = load_reference_sticks(sorted(REFERENCE_DIR.glob("*.cif")))
    all_formulas = sorted(refs_by_formula.keys())

    X, y_label_lists = build_synthetic_multiphase_dataset(refs_by_formula, two_theta_grid, rng)

    mlb = MultiLabelBinarizer(classes=all_formulas)
    y_bin = mlb.fit_transform(y_label_lists)

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y_bin,
        test_size=VAL_FRACTION,
        random_state=RANDOM_SEED,
    )

    exp_files = sorted(EXPERIMENT_DIR.glob("*.xy"))
    X_test = np.asarray([preprocess_experimental_pattern(f, two_theta_grid) for f in exp_files])
    y_test_labels = [labels_from_filename(f.stem) for f in exp_files]
    y_test = mlb.transform(y_test_labels)

    print("\n=== Multi-Phase Conventional ML Demo (k-NN, Random Forest, SVM) ===")
    print(f"Synthetic samples: {len(X)}")
    print(f"Unique formulas (labels): {len(all_formulas)}")
    print(f"Training samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    print(f"Experimental test patterns: {len(X_test)}")

    model_results = {
        "k-NN": tune_model(
            X_train,
            y_train,
            X_val,
            y_val,
            model_builder=lambda n_neighbors: OneVsRestClassifier(
                make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=n_neighbors, weights="distance"))
            ),
            param_grid=[{"n_neighbors": k} for k in [1, 3, 5, 7, 11]],
        ),
        "Random Forest": tune_model(
            X_train,
            y_train,
            X_val,
            y_val,
            model_builder=lambda n_estimators, max_depth, min_samples_leaf: OneVsRestClassifier(
                RandomForestClassifier(
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    min_samples_leaf=min_samples_leaf,
                    random_state=RANDOM_SEED,
                    n_jobs=-1,
                )
            ),
            param_grid=[
                {"n_estimators": 250, "max_depth": d, "min_samples_leaf": l}
                for d in [None, 15, 30]
                for l in [1, 2, 4]
            ],
        ),
        "SVM": tune_model(
            X_train,
            y_train,
            X_val,
            y_val,
            model_builder=lambda C, gamma: OneVsRestClassifier(
                make_pipeline(StandardScaler(), SVC(kernel="rbf", C=C, gamma=gamma, probability=True))
            ),
            param_grid=[{"C": c, "gamma": g} for c in [1.0, 5.0, 10.0] for g in ["scale", 0.01, 0.001]],
        ),
    }

    prediction_rows = []

    for model_name, result in model_results.items():
        model = result["model"]
        threshold = result["threshold"]
        test_scores = get_label_scores(model, X_test)
        test_precision_micro, test_recall_micro, test_f1_micro, y_pred_test = threshold_metrics(y_test, test_scores, threshold)

        result["test_precision_micro"] = test_precision_micro
        result["test_recall_micro"] = test_recall_micro
        result["test_f1_micro"] = test_f1_micro

        print(f"\n{model_name}")
        print(f"  Best params: {result['params']}")
        print(f"  Validation-picked threshold: {threshold:.2f}")
        print(f"  Test precision (micro): {test_precision_micro:.3f}")
        print(f"  Test recall (micro): {test_recall_micro:.3f}")
        print(f"  Test F1 (micro): {test_f1_micro:.3f}")

        print("  Test predictions (threshold-based):")
        for i, exp_file in enumerate(exp_files):
            true_set = y_test_labels[i]
            pred_set = list(mlb.classes_[np.where(y_pred_test[i] == 1)[0]])
            inter = len(set(true_set) & set(pred_set))
            f1_pattern = 0.0 if (len(true_set) + len(pred_set)) == 0 else 2.0 * inter / (len(true_set) + len(pred_set))

            print(f"    {exp_file.stem:26s} -> {pred_set}")
            prediction_rows.append(
                {
                    "model": model_name,
                    "pattern": exp_file.stem,
                    "true_labels": ";".join(true_set),
                    "predicted_labels": ";".join(pred_set),
                    "n_predicted": len(pred_set),
                    "threshold": threshold,
                    "f1_pattern": f1_pattern,
                }
            )

    plot_test_metric_summary(model_results)

    metrics_file = OUTPUT_DIR / "model_test_metrics.csv"
    with open(metrics_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["model", "threshold", "test_precision_micro", "test_recall_micro", "test_f1_micro", "best_params"],
        )
        writer.writeheader()
        for name, result in model_results.items():
            writer.writerow(
                {
                    "model": name,
                    "threshold": result["threshold"],
                    "test_precision_micro": result["test_precision_micro"],
                    "test_recall_micro": result["test_recall_micro"],
                    "test_f1_micro": result["test_f1_micro"],
                    "best_params": str(result["params"]),
                }
            )

    pred_file = OUTPUT_DIR / "test_predictions_threshold.csv"
    with open(pred_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["model", "pattern", "true_labels", "predicted_labels", "n_predicted", "threshold", "f1_pattern"],
        )
        writer.writeheader()
        writer.writerows(prediction_rows)

    print(f"\nSaved metrics: {metrics_file}")
    print(f"Saved predictions: {pred_file}")


# 03b — Conventional ML: Multiphase Classification

This notebook extends to multi-label phase prediction using synthetic mixtures and thresholded outputs.

## Runtime Note
The dataset size is reduced for in-session runtime. Increase sample counts for stronger offline models.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
SYNTH_SAMPLES_PER_FORMULA = 10  # Reduced for Colab demo speed
main()


## What To Observe
Focus on precision/recall/F1 tradeoffs after threshold selection.

In [ ]:
display(Image("outputs/ml/multiphase/model_test-metric_summary.png"))

## Summary
- Multiphase ID is naturally a multi-label problem.
- Threshold tuning materially changes precision-recall behavior.
- Synthetic mixture design controls model robustness.

## Next Steps
Continue to the next section below in this notebook.

## 04a - NNs 1phase

In [ ]:
from IPython.display import Image, display

# Inline tutorial script
from pathlib import Path
import csv

# For handling arrays
import numpy as np

# For plotting
import matplotlib.pyplot as plt

# To load structures and compute XRD stick patterns
from pymatgen.core import Structure
from pymatgen.analysis.diffraction.xrd import XRDCalculator

# Neural-network model
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score


# Input/output
EXPERIMENT_DIR = Path("data/exp_patterns/one_phase")
REFERENCE_DIR = Path("data/reference_structures")
OUTPUT_DIR = Path("outputs/dl/nn_1phase")

# Pattern settings
MIN_ANGLE = 10.0
MAX_ANGLE = 80.0
NUM_POINTS = 1800
WAVELENGTH = "CuKa"
WAVELENGTH_ANGSTROM = 1.5406
REFERENCE_INTENSITY_THRESHOLD = 1.0

# Synthetic-data settings
SYNTH_SAMPLES_PER_FORMULA = 80
RANDOM_SEED = 42

# Neural-net settings (fixed architecture; no tuning)
NN_HIDDEN_LAYER_SIZES = (128, 64)
NN_ALPHA = 1e-4
NN_LEARNING_RATE_INIT = 1e-3
NN_MAX_ITER = 220

# Artifact ranges (same style as Slide-34)
UNIFORM_SHIFT_RANGE = (-0.15, 0.15)
SAMPLE_DISPLACEMENT_RANGE_MM = (-0.20, 0.20)
GONIOMETER_RADIUS_MM = 240.0

U_RANGE = (0.01, 0.06)
V_RANGE = (-0.02, 0.01)
W_RANGE = (0.002, 0.010)
SIZE_NM_RANGE = (8.0, 80.0)
MICROSTRAIN_RANGE = (0.0, 0.003)

FWHM_RANGE = (0.08, 0.45)
ETA_RANGE = (0.10, 0.70)

BACKGROUND_SCALE_RANGE = (0.05, 0.30)
HUMP_SCALE_RANGE = (0.02, 0.20)
NOISE_SCALE_RANGE = (0.002, 0.020)

# Split
VAL_FRACTION = 0.20


# -----------------------------
# Pattern simulation utilities
# -----------------------------
def normalize_0_100(y):
    y = np.asarray(y, dtype=float)
    y = y - y.min()
    return 100.0 * y / np.clip(y.max(), 1e-12, None)


def sample_displacement_shift(two_theta_deg, displacement_mm):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    d_relative_change = displacement_mm / GONIOMETER_RADIUS_MM * np.cos(theta_rad) ** 2
    return np.rad2deg(-d_relative_change * np.tan(theta_rad))


def instrumental_fwhm(two_theta_deg, u, v, w):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    tan_theta = np.tan(theta_rad)
    fwhm_sq = u * tan_theta**2 + v * tan_theta + w
    return np.sqrt(np.clip(fwhm_sq, 1e-4, None))


def size_fwhm(two_theta_deg, size_nm):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    wavelength_nm = WAVELENGTH_ANGSTROM / 10.0
    beta_rad = 0.9 * wavelength_nm / (size_nm * np.cos(theta_rad))
    return np.rad2deg(beta_rad)


def strain_fwhm(two_theta_deg, microstrain):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    beta_rad = 4.0 * microstrain * np.tan(theta_rad)
    return np.rad2deg(beta_rad)


def pseudo_voigt_profile(two_theta_grid, centers, fwhm, eta):
    dx = two_theta_grid[:, None] - centers[None, :]
    sigma = np.clip(fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0))), 1e-6, None)
    gamma = np.clip(fwhm / 2.0, 1e-6, None)
    gauss = np.exp(-0.5 * (dx / sigma[None, :]) ** 2)
    lorentz = (gamma[None, :] ** 2) / (dx**2 + gamma[None, :] ** 2)
    return (1.0 - eta) * gauss + eta * lorentz


def load_reference_sticks(cif_files):
    calc = XRDCalculator(wavelength=WAVELENGTH)
    refs = []

    for cif_file in cif_files:
        pattern = calc.get_pattern(Structure.from_file(cif_file), two_theta_range=(MIN_ANGLE, MAX_ANGLE))
        peak_pos = np.asarray(pattern.x, dtype=float)
        peak_int = np.asarray(pattern.y, dtype=float)

        keep = peak_int >= REFERENCE_INTENSITY_THRESHOLD
        peak_pos = peak_pos[keep]
        peak_int = normalize_0_100(peak_int[keep])

        formula = cif_file.stem.split("_", 1)[0]
        refs.append({"phase": cif_file.stem, "formula": formula, "peak_pos": peak_pos, "peak_int": peak_int})

    return refs


def simulate_artifact_profile(two_theta_grid, base_pos, base_int, rng):
    if len(base_pos) == 0:
        return np.zeros_like(two_theta_grid)

    peak_int = base_int * np.exp(rng.normal(0.0, 0.25, size=len(base_int)))

    uniform_shift = rng.uniform(*UNIFORM_SHIFT_RANGE)
    displacement = rng.uniform(*SAMPLE_DISPLACEMENT_RANGE_MM)
    peak_pos = base_pos + uniform_shift + sample_displacement_shift(base_pos, displacement)

    keep = (peak_pos >= MIN_ANGLE - 1.0) & (peak_pos <= MAX_ANGLE + 1.0)
    peak_pos = peak_pos[keep]
    peak_int = peak_int[keep]
    if len(peak_pos) == 0:
        return np.zeros_like(two_theta_grid)

    u = rng.uniform(*U_RANGE)
    v = rng.uniform(*V_RANGE)
    w = rng.uniform(*W_RANGE)
    size_nm = rng.uniform(*SIZE_NM_RANGE)
    microstrain = rng.uniform(*MICROSTRAIN_RANGE)

    fwhm = np.sqrt(
        instrumental_fwhm(peak_pos, u, v, w) ** 2
        + size_fwhm(peak_pos, size_nm) ** 2
        + strain_fwhm(peak_pos, microstrain) ** 2
    )
    fwhm += rng.uniform(*FWHM_RANGE)
    eta = rng.uniform(*ETA_RANGE)

    peaks = pseudo_voigt_profile(two_theta_grid, peak_pos, fwhm, eta) @ peak_int
    peaks = normalize_0_100(peaks)

    x_cheb = 2.0 * (two_theta_grid - MIN_ANGLE) / (MAX_ANGLE - MIN_ANGLE) - 1.0
    coeffs = np.array([1.0, rng.uniform(-0.5, 0.5), rng.uniform(-0.4, 0.4), rng.uniform(-0.2, 0.2), rng.uniform(-0.1, 0.1)])
    background = np.polynomial.chebyshev.chebval(x_cheb, coeffs)
    background -= background.min()
    background /= np.clip(background.max(), 1e-12, None)
    background *= rng.uniform(*BACKGROUND_SCALE_RANGE) * peaks.max()

    center = rng.uniform(18.0, 35.0)
    width = rng.uniform(5.0, 12.0)
    hump = rng.uniform(*HUMP_SCALE_RANGE) * peaks.max() * np.exp(-0.5 * ((two_theta_grid - center) / width) ** 2)

    noise_sigma = rng.uniform(*NOISE_SCALE_RANGE) * peaks.max()
    noise = rng.normal(0.0, noise_sigma, size=len(two_theta_grid))

    y = peaks + background + hump + noise
    y -= y.min()
    return normalize_0_100(y)


def build_synthetic_dataset(reference_sticks, two_theta_grid, rng):
    grouped = {}
    for ref in reference_sticks:
        grouped.setdefault(ref["formula"], []).append(ref)

    X = []
    y = []

    for formula, entries in sorted(grouped.items()):
        for _ in range(SYNTH_SAMPLES_PER_FORMULA):
            ref = entries[rng.integers(0, len(entries))]
            profile = simulate_artifact_profile(two_theta_grid, ref["peak_pos"], ref["peak_int"], rng)
            X.append(profile)
            y.append(formula)

    return np.asarray(X), np.asarray(y)


def preprocess_experimental_pattern(xy_file, two_theta_grid):
    data = np.loadtxt(xy_file)
    x = data[:, 0]
    y = data[:, 1]

    keep = (x >= MIN_ANGLE) & (x <= MAX_ANGLE)
    x = x[keep]
    y = y[keep]

    y_interp = np.interp(two_theta_grid, x, y)
    return normalize_0_100(y_interp)


# -----------------------------
# Neural-network training
# -----------------------------
def build_nn():
    return make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=NN_HIDDEN_LAYER_SIZES,
            activation="relu",
            solver="adam",
            alpha=NN_ALPHA,
            batch_size="auto",
            learning_rate_init=NN_LEARNING_RATE_INIT,
            max_iter=NN_MAX_ITER,
            random_state=RANDOM_SEED,
        ),
    )


def plot_loss_curve(train_loss_curve):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    epochs = np.arange(1, len(train_loss_curve) + 1)

    ax.plot(epochs, train_loss_curve, color="tab:blue", linewidth=2.2, label="Train")

    ax.set_xlabel("Epoch", fontsize=18, labelpad=8)
    ax.set_ylabel("Loss", fontsize=18, labelpad=10)
    ax.tick_params(axis="both", labelsize=16)
    ax.legend(fontsize=16, loc="upper right", framealpha=1)
    ax.grid(alpha=0.25)

    out_file = OUTPUT_DIR / "nn_loss_curve.png"
    plt.tight_layout()
    plt.savefig(out_file, dpi=200)
    plt.close(fig)
    print(f"Saved plot: {out_file}")


def plot_accuracy_summary(val_acc, test_acc):
    fig, ax = plt.subplots(figsize=(6.4, 4.5))

    x = np.array([0])
    width = 0.36
    ax.bar(x - width / 2, [val_acc], width=width, color="tab:blue", edgecolor="black", linewidth=1.0, label="Validation")
    ax.bar(x + width / 2, [test_acc], width=width, color="tab:red", edgecolor="black", linewidth=1.0, label="Test")

    ax.set_xticks(x)
    ax.set_xticklabels(["Neural Net"], fontsize=16)
    ax.tick_params(axis="y", labelsize=16)
    ax.set_ylim(0.0, 1.05)
    ax.set_ylabel("Accuracy", fontsize=18, labelpad=12)
    ax.legend(fontsize=16, loc="lower right", framealpha=1)
    ax.grid(axis="y", alpha=0.25)

    out_file = OUTPUT_DIR / "nn_accuracy_summary.png"
    plt.tight_layout()
    plt.savefig(out_file, dpi=200)
    plt.close(fig)
    print(f"Saved plot: {out_file}")


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(RANDOM_SEED)
    two_theta_grid = np.linspace(MIN_ANGLE, MAX_ANGLE, NUM_POINTS)

    reference_sticks = load_reference_sticks(sorted(REFERENCE_DIR.glob("*.cif")))
    X, y = build_synthetic_dataset(reference_sticks, two_theta_grid, rng)

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=VAL_FRACTION,
        random_state=RANDOM_SEED,
        stratify=y,
    )

    exp_files = sorted(EXPERIMENT_DIR.glob("*.xy"))
    X_test = np.asarray([preprocess_experimental_pattern(f, two_theta_grid) for f in exp_files])
    y_test = np.asarray([f.stem for f in exp_files])

    print("\n=== Neural-Net Phase-ID Demo (1-phase) ===")
    print(f"Synthetic training+validation samples: {len(X)}")
    print(f"Number of phase classes: {len(np.unique(y))}")
    print(f"Training samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    print(f"Test patterns (experimental): {len(X_test)}")

    model = build_nn()
    model.fit(X_train, y_train)

    val_acc = accuracy_score(y_val, model.predict(X_val))
    y_pred_test = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred_test)
    train_loss_curve = np.asarray(model.named_steps["mlpclassifier"].loss_curve_, dtype=float)

    print("\nNeural Net")
    print(f"  Fixed hidden layers: {NN_HIDDEN_LAYER_SIZES}")
    print(f"  Output nodes (classes): {len(np.unique(y))}")
    print(f"  Validation accuracy: {val_acc:.3f}")
    print(f"  Test accuracy: {test_acc:.3f}")

    print("  Test predictions:")
    prediction_rows = []
    for true_label, pred_label in zip(y_test, y_pred_test):
        correct = true_label == pred_label
        mark = "\u2713" if correct else "x"
        print(f"    {true_label:8s} -> {pred_label:8s} {mark}")
        prediction_rows.append(
            {
                "pattern": true_label,
                "true_phase": true_label,
                "predicted_phase": pred_label,
                "correct": int(correct),
            }
        )

    if len(train_loss_curve) > 0:
        plot_loss_curve(train_loss_curve)
    plot_accuracy_summary(val_acc, test_acc)

    metrics_file = OUTPUT_DIR / "nn_metrics.csv"
    with open(metrics_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["model", "hidden_layers", "n_outputs", "validation_accuracy", "test_accuracy"],
        )
        writer.writeheader()
        writer.writerow(
            {
                "model": "Neural Net",
                "hidden_layers": str(NN_HIDDEN_LAYER_SIZES),
                "n_outputs": len(np.unique(y)),
                "validation_accuracy": val_acc,
                "test_accuracy": test_acc,
            }
        )

    pred_file = OUTPUT_DIR / "experimental_predictions.csv"
    with open(pred_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["pattern", "true_phase", "predicted_phase", "correct"],
        )
        writer.writeheader()
        writer.writerows(prediction_rows)

    print(f"\nSaved metrics: {metrics_file}")
    print(f"Saved predictions: {pred_file}")


# 04a — Neural Networks: 1-Phase

A feedforward neural network is trained on synthetic 1-phase profiles and evaluated on experimental patterns.

## Runtime Note
`NN_MAX_ITER` is reduced here for live speed. Increase it for full convergence offline.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
SYNTH_SAMPLES_PER_FORMULA = 12
NN_MAX_ITER = 80  # Reduced for demo speed
main()


## What To Observe
Inspect the training loss trend and compare validation vs test accuracy.

In [ ]:
display(Image("outputs/dl/nn_1phase/nn_loss_curve.png"))
display(Image("outputs/dl/nn_1phase/nn_accuracy_summary.png"))

## Summary
- Simple dense NNs can capture nonlinear XRD-feature interactions.
- Training duration strongly affects final accuracy.
- Synthetic realism still matters as much as network choice.

## Next Steps
Continue to the next section below in this notebook.

## 04b - NNs Multiphase

In [ ]:
from IPython.display import Image, display

# Inline tutorial script
from pathlib import Path
import csv

# For handling arrays
import numpy as np

# For plotting
import matplotlib.pyplot as plt

# To load structures and compute XRD stick patterns
from pymatgen.core import Structure
from pymatgen.analysis.diffraction.xrd import XRDCalculator

# Neural-network model
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import precision_score, recall_score, f1_score


# Input/output
EXPERIMENT_DIR = Path("data/exp_patterns/multi_phase")
REFERENCE_DIR = Path("data/reference_structures")
OUTPUT_DIR = Path("outputs/dl/nn_multiphase")

# Pattern settings
MIN_ANGLE = 10.0
MAX_ANGLE = 80.0
NUM_POINTS = 1400
WAVELENGTH = "CuKa"
WAVELENGTH_ANGSTROM = 1.5406
REFERENCE_INTENSITY_THRESHOLD = 1.0

# Synthetic-data settings
SYNTH_SAMPLES_PER_FORMULA = 60
RANDOM_SEED = 42
SINGLE_PHASE_FRACTION = 0.15

# Neural-net settings (fixed architecture; no tuning)
NN_HIDDEN_LAYER_SIZES = (128, 64)
NN_ALPHA = 1e-4
NN_LEARNING_RATE_INIT = 1e-3
NN_MAX_ITER = 220
PREDICTION_THRESHOLD = 0.50

# Artifact ranges
UNIFORM_SHIFT_RANGE = (-0.12, 0.12)
SAMPLE_DISPLACEMENT_RANGE_MM = (-0.18, 0.18)
GONIOMETER_RADIUS_MM = 240.0

U_RANGE = (0.01, 0.06)
V_RANGE = (-0.02, 0.01)
W_RANGE = (0.002, 0.010)
SIZE_NM_RANGE = (8.0, 80.0)
MICROSTRAIN_RANGE = (0.0, 0.003)

FWHM_RANGE = (0.08, 0.45)
ETA_RANGE = (0.10, 0.70)

BACKGROUND_SCALE_RANGE = (0.05, 0.28)
HUMP_SCALE_RANGE = (0.02, 0.20)
NOISE_SCALE_RANGE = (0.002, 0.020)

# Split
VAL_FRACTION = 0.20


# -----------------------------
# Pattern simulation utilities
# -----------------------------
def normalize_0_100(y):
    y = np.asarray(y, dtype=float)
    y = y - y.min()
    return 100.0 * y / np.clip(y.max(), 1e-12, None)


def sample_displacement_shift(two_theta_deg, displacement_mm):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    d_relative_change = displacement_mm / GONIOMETER_RADIUS_MM * np.cos(theta_rad) ** 2
    return np.rad2deg(-d_relative_change * np.tan(theta_rad))


def instrumental_fwhm(two_theta_deg, u, v, w):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    tan_theta = np.tan(theta_rad)
    fwhm_sq = u * tan_theta**2 + v * tan_theta + w
    return np.sqrt(np.clip(fwhm_sq, 1e-4, None))


def size_fwhm(two_theta_deg, size_nm):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    wavelength_nm = WAVELENGTH_ANGSTROM / 10.0
    beta_rad = 0.9 * wavelength_nm / (size_nm * np.cos(theta_rad))
    return np.rad2deg(beta_rad)


def strain_fwhm(two_theta_deg, microstrain):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    beta_rad = 4.0 * microstrain * np.tan(theta_rad)
    return np.rad2deg(beta_rad)


def pseudo_voigt_profile(two_theta_grid, centers, fwhm, eta):
    dx = two_theta_grid[:, None] - centers[None, :]
    sigma = np.clip(fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0))), 1e-6, None)
    gamma = np.clip(fwhm / 2.0, 1e-6, None)
    gauss = np.exp(-0.5 * (dx / sigma[None, :]) ** 2)
    lorentz = (gamma[None, :] ** 2) / (dx**2 + gamma[None, :] ** 2)
    return (1.0 - eta) * gauss + eta * lorentz


def load_reference_sticks(cif_files):
    calc = XRDCalculator(wavelength=WAVELENGTH)
    refs_by_formula = {}

    for cif_file in cif_files:
        pattern = calc.get_pattern(Structure.from_file(cif_file), two_theta_range=(MIN_ANGLE, MAX_ANGLE))
        peak_pos = np.asarray(pattern.x, dtype=float)
        peak_int = np.asarray(pattern.y, dtype=float)

        keep = peak_int >= REFERENCE_INTENSITY_THRESHOLD
        peak_pos = peak_pos[keep]
        peak_int = normalize_0_100(peak_int[keep])

        formula = cif_file.stem.split("_", 1)[0]
        refs_by_formula.setdefault(formula, []).append({"phase": cif_file.stem, "peak_pos": peak_pos, "peak_int": peak_int})

    return refs_by_formula


def simulate_component_profile(two_theta_grid, base_pos, base_int, rng):
    if len(base_pos) == 0:
        return np.zeros_like(two_theta_grid)

    peak_int = base_int * np.exp(rng.normal(0.0, 0.25, size=len(base_int)))

    uniform_shift = rng.uniform(*UNIFORM_SHIFT_RANGE)
    displacement = rng.uniform(*SAMPLE_DISPLACEMENT_RANGE_MM)
    peak_pos = base_pos + uniform_shift + sample_displacement_shift(base_pos, displacement)

    keep = (peak_pos >= MIN_ANGLE - 1.0) & (peak_pos <= MAX_ANGLE + 1.0)
    peak_pos = peak_pos[keep]
    peak_int = peak_int[keep]
    if len(peak_pos) == 0:
        return np.zeros_like(two_theta_grid)

    u = rng.uniform(*U_RANGE)
    v = rng.uniform(*V_RANGE)
    w = rng.uniform(*W_RANGE)
    size_nm = rng.uniform(*SIZE_NM_RANGE)
    microstrain = rng.uniform(*MICROSTRAIN_RANGE)

    fwhm = np.sqrt(
        instrumental_fwhm(peak_pos, u, v, w) ** 2
        + size_fwhm(peak_pos, size_nm) ** 2
        + strain_fwhm(peak_pos, microstrain) ** 2
    )
    fwhm += rng.uniform(*FWHM_RANGE)
    eta = rng.uniform(*ETA_RANGE)

    profile = pseudo_voigt_profile(two_theta_grid, peak_pos, fwhm, eta) @ peak_int
    return profile / np.clip(profile.max(), 1e-12, None)


def simulate_multiphase_profile(two_theta_grid, formula_list, refs_by_formula, rng):
    components = []
    for formula in formula_list:
        ref = refs_by_formula[formula][rng.integers(0, len(refs_by_formula[formula]))]
        components.append(simulate_component_profile(two_theta_grid, ref["peak_pos"], ref["peak_int"], rng))

    weights = rng.dirichlet(np.ones(len(components)) * 1.5)
    peaks = np.zeros_like(two_theta_grid)
    for w, comp in zip(weights, components):
        peaks += w * comp
    peaks = normalize_0_100(peaks)

    x_cheb = 2.0 * (two_theta_grid - MIN_ANGLE) / (MAX_ANGLE - MIN_ANGLE) - 1.0
    coeffs = np.array([1.0, rng.uniform(-0.5, 0.5), rng.uniform(-0.4, 0.4), rng.uniform(-0.2, 0.2), rng.uniform(-0.1, 0.1)])
    background = np.polynomial.chebyshev.chebval(x_cheb, coeffs)
    background -= background.min()
    background /= np.clip(background.max(), 1e-12, None)
    background *= rng.uniform(*BACKGROUND_SCALE_RANGE) * peaks.max()

    center = rng.uniform(18.0, 35.0)
    width = rng.uniform(5.0, 12.0)
    hump = rng.uniform(*HUMP_SCALE_RANGE) * peaks.max() * np.exp(-0.5 * ((two_theta_grid - center) / width) ** 2)

    noise_sigma = rng.uniform(*NOISE_SCALE_RANGE) * peaks.max()
    noise = rng.normal(0.0, noise_sigma, size=len(two_theta_grid))

    y = peaks + background + hump + noise
    y -= y.min()
    return normalize_0_100(y)


def build_synthetic_multiphase_dataset(refs_by_formula, two_theta_grid, rng):
    formulas = sorted(refs_by_formula.keys())
    X = []
    y_labels = []

    for anchor in formulas:
        others = [f for f in formulas if f != anchor]
        for _ in range(SYNTH_SAMPLES_PER_FORMULA):
            n_components = int(
                rng.choice(
                    [1, 2, 3],
                    p=[
                        SINGLE_PHASE_FRACTION,
                        (1.0 - SINGLE_PHASE_FRACTION) * 0.65,
                        (1.0 - SINGLE_PHASE_FRACTION) * 0.35,
                    ],
                )
            )
            chosen_others = list(rng.choice(others, size=n_components - 1, replace=False))
            labels = sorted([anchor] + chosen_others)

            profile = simulate_multiphase_profile(two_theta_grid, labels, refs_by_formula, rng)
            X.append(profile)
            y_labels.append(labels)

    return np.asarray(X), y_labels


def preprocess_experimental_pattern(xy_file, two_theta_grid):
    data = np.loadtxt(xy_file)
    x = data[:, 0]
    y = data[:, 1]

    keep = (x >= MIN_ANGLE) & (x <= MAX_ANGLE)
    x = x[keep]
    y = y[keep]

    y_interp = np.interp(two_theta_grid, x, y)
    return normalize_0_100(y_interp)


# -----------------------------
# Multi-label NN utilities
# -----------------------------
def labels_from_filename(file_stem):
    return file_stem.split("_")


def build_simple_nn():
    return make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=NN_HIDDEN_LAYER_SIZES,
            activation="relu",
            solver="adam",
            alpha=NN_ALPHA,
            batch_size="auto",
            learning_rate_init=NN_LEARNING_RATE_INIT,
            max_iter=NN_MAX_ITER,
            random_state=RANDOM_SEED,
        ),
    )


def get_label_scores(model, X):
    if hasattr(model, "predict_proba"):
        scores = model.predict_proba(X)
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X)
    else:
        scores = model.predict(X)

    if isinstance(scores, list):
        cols = []
        for s in scores:
            s = np.asarray(s)
            if s.ndim == 2 and s.shape[1] == 2:
                cols.append(s[:, 1])
            else:
                cols.append(s.ravel())
        return np.column_stack(cols)

    scores = np.asarray(scores)
    if scores.ndim == 1:
        scores = scores[:, None]
    return scores


def predict_binary_labels(scores, threshold=PREDICTION_THRESHOLD):
    return (scores >= threshold).astype(int)


def compute_micro_metrics(y_true_bin, y_pred_bin):
    precision_micro = precision_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    recall_micro = recall_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    f1_micro = f1_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    return precision_micro, recall_micro, f1_micro


def plot_loss_curve(train_loss_curve):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    epochs = np.arange(1, len(train_loss_curve) + 1)

    ax.plot(epochs, train_loss_curve, color="tab:blue", linewidth=2.2, label="Train")

    ax.set_xlabel("Epoch", fontsize=18, labelpad=8)
    ax.set_ylabel("Loss", fontsize=18, labelpad=10)
    ax.tick_params(axis="both", labelsize=16)
    ax.legend(fontsize=16, loc="upper right", framealpha=1)
    ax.grid(alpha=0.25)

    out_file = OUTPUT_DIR / "nn_loss_curve.png"
    plt.tight_layout()
    plt.savefig(out_file, dpi=200)
    plt.close(fig)
    print(f"Saved plot: {out_file}")


def plot_test_metric_summary(precision_micro, recall_micro, f1_micro):
    fig, ax = plt.subplots(figsize=(6.6, 4.5))

    x = np.array([0])
    width = 0.24
    ax.bar(x - width, [precision_micro], width=width, color="tab:blue", edgecolor="black", linewidth=1.0, label="Precision")
    ax.bar(x, [recall_micro], width=width, color="tab:green", edgecolor="black", linewidth=1.0, label="Recall")
    ax.bar(x + width, [f1_micro], width=width, color="tab:red", edgecolor="black", linewidth=1.0, label="F1-score")

    ax.set_xticks(x)
    ax.set_xticklabels(["Neural Net"], fontsize=16)
    ax.tick_params(axis="y", labelsize=16)
    ax.set_ylim(0.0, 1.05)
    ax.set_ylabel("Score", fontsize=18, labelpad=12)
    ax.legend(fontsize=16, loc="lower right", framealpha=1)
    ax.grid(axis="y", alpha=0.25)

    out_file = OUTPUT_DIR / "nn_test-metric_summary.png"
    plt.tight_layout()
    plt.savefig(out_file, dpi=200)
    plt.close(fig)
    print(f"Saved plot: {out_file}")


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(RANDOM_SEED)
    two_theta_grid = np.linspace(MIN_ANGLE, MAX_ANGLE, NUM_POINTS)

    refs_by_formula = load_reference_sticks(sorted(REFERENCE_DIR.glob("*.cif")))
    all_formulas = sorted(refs_by_formula.keys())

    X, y_label_lists = build_synthetic_multiphase_dataset(refs_by_formula, two_theta_grid, rng)

    mlb = MultiLabelBinarizer(classes=all_formulas)
    y_bin = mlb.fit_transform(y_label_lists)

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y_bin,
        test_size=VAL_FRACTION,
        random_state=RANDOM_SEED,
    )

    exp_files = sorted(EXPERIMENT_DIR.glob("*.xy"))
    X_test = np.asarray([preprocess_experimental_pattern(f, two_theta_grid) for f in exp_files])
    y_test_labels = [labels_from_filename(f.stem) for f in exp_files]
    y_test = mlb.transform(y_test_labels)

    print("\n=== Neural-Net Phase-ID Demo (multi-phase) ===")
    print(f"Synthetic samples: {len(X)}")
    print(f"Unique formulas (labels): {len(all_formulas)}")
    print(f"Training samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    print(f"Experimental test patterns: {len(X_test)}")

    model = build_simple_nn()
    model.fit(X_train, y_train)

    val_scores = get_label_scores(model, X_val)
    y_pred_val = predict_binary_labels(val_scores, threshold=PREDICTION_THRESHOLD)
    val_precision_micro, val_recall_micro, val_f1_micro = compute_micro_metrics(y_val, y_pred_val)

    test_scores = get_label_scores(model, X_test)
    y_pred_test = predict_binary_labels(test_scores, threshold=PREDICTION_THRESHOLD)
    test_precision_micro, test_recall_micro, test_f1_micro = compute_micro_metrics(y_test, y_pred_test)

    mlp = model.named_steps["mlpclassifier"]
    train_loss_curve = np.asarray(getattr(mlp, "loss_curve_", []), dtype=float)

    print("\nNeural Net")
    print(f"  Fixed hidden layers: {NN_HIDDEN_LAYER_SIZES}")
    print(f"  Output nodes (labels): {len(all_formulas)}")
    print(f"  Binary threshold: {PREDICTION_THRESHOLD:.2f}")
    print(f"  Validation precision (micro): {val_precision_micro:.3f}")
    print(f"  Validation recall (micro): {val_recall_micro:.3f}")
    print(f"  Validation F1 (micro): {val_f1_micro:.3f}")
    print(f"  Test precision (micro): {test_precision_micro:.3f}")
    print(f"  Test recall (micro): {test_recall_micro:.3f}")
    print(f"  Test F1 (micro): {test_f1_micro:.3f}")

    print("  Test predictions (binary outputs):")
    prediction_rows = []
    for i, exp_file in enumerate(exp_files):
        true_set = y_test_labels[i]
        pred_set = list(mlb.classes_[np.where(y_pred_test[i] == 1)[0]])

        true_binary = " ".join(map(str, y_test[i].astype(int).tolist()))
        pred_binary = " ".join(map(str, y_pred_test[i].astype(int).tolist()))
        inter = len(set(true_set) & set(pred_set))
        f1_pattern = 0.0 if (len(true_set) + len(pred_set)) == 0 else 2.0 * inter / (len(true_set) + len(pred_set))

        print(f"    {exp_file.stem:26s} -> [{pred_binary}] {pred_set}")
        prediction_rows.append(
            {
                "pattern": exp_file.stem,
                "true_labels": ";".join(true_set),
                "predicted_labels": ";".join(pred_set),
                "true_binary": true_binary,
                "predicted_binary": pred_binary,
                "n_predicted": len(pred_set),
                "f1_pattern": f1_pattern,
            }
        )

    if len(train_loss_curve) > 0:
        plot_loss_curve(train_loss_curve)
    plot_test_metric_summary(test_precision_micro, test_recall_micro, test_f1_micro)

    metrics_file = OUTPUT_DIR / "nn_metrics.csv"
    with open(metrics_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "model",
                "hidden_layers",
                "n_outputs",
                "threshold",
                "validation_precision_micro",
                "validation_recall_micro",
                "validation_f1_micro",
                "test_precision_micro",
                "test_recall_micro",
                "test_f1_micro",
            ],
        )
        writer.writeheader()
        writer.writerow(
            {
                "model": "Neural Net",
                "hidden_layers": str(NN_HIDDEN_LAYER_SIZES),
                "n_outputs": len(all_formulas),
                "threshold": PREDICTION_THRESHOLD,
                "validation_precision_micro": val_precision_micro,
                "validation_recall_micro": val_recall_micro,
                "validation_f1_micro": val_f1_micro,
                "test_precision_micro": test_precision_micro,
                "test_recall_micro": test_recall_micro,
                "test_f1_micro": test_f1_micro,
            }
        )

    pred_file = OUTPUT_DIR / "test_predictions_binary.csv"
    with open(pred_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "pattern",
                "true_labels",
                "predicted_labels",
                "true_binary",
                "predicted_binary",
                "n_predicted",
                "f1_pattern",
            ],
        )
        writer.writeheader()
        writer.writerows(prediction_rows)

    print(f"\nSaved metrics: {metrics_file}")
    print(f"Saved predictions: {pred_file}")


# 04b — Neural Networks: Multiphase

We apply a dense multi-label NN to mixed-phase data and evaluate micro-averaged metrics.

## Runtime Note
This run uses reduced synthetic data and iterations to keep in-class runtime manageable.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
SYNTH_SAMPLES_PER_FORMULA = 8
NN_MAX_ITER = 80  # Reduced for demo speed
main()


## What To Observe
Check whether recall and precision remain balanced at the fixed threshold.

In [ ]:
display(Image("outputs/dl/nn_multiphase/nn_loss_curve.png"))
display(Image("outputs/dl/nn_multiphase/nn_test-metric_summary.png"))

## Summary
- Dense NNs can perform multi-label phase inference.
- Threshold choice is critical for practical decoding.
- Model capacity and training data diversity must be balanced.

## Next Steps
Continue to the next section below in this notebook.

## 04c - CNNs Multiphase

In [ ]:
import os
from IPython.display import Image, display

# Inline tutorial script
from pathlib import Path
import csv

# For handling arrays
import numpy as np

# For plotting
import matplotlib.pyplot as plt

# To load structures and compute XRD stick patterns
from pymatgen.core import Structure
from pymatgen.analysis.diffraction.xrd import XRDCalculator

# Neural-network model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.metrics import precision_score, recall_score, f1_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


# Input/output
EXPERIMENT_DIR = Path("data/exp_patterns/multi_phase")
REFERENCE_DIR = Path("data/reference_structures")
OUTPUT_DIR = Path("outputs/dl/cnn_multiphase")

# Pattern settings
MIN_ANGLE = 10.0
MAX_ANGLE = 80.0
NUM_POINTS = 1400
WAVELENGTH = "CuKa"
WAVELENGTH_ANGSTROM = 1.5406
REFERENCE_INTENSITY_THRESHOLD = 1.0

# Synthetic-data settings
SYNTH_SAMPLES_PER_FORMULA = 60
RANDOM_SEED = 42
SINGLE_PHASE_FRACTION = 0.15

# Neural-net settings (fixed architecture; no tuning)
CNN_CONV_CHANNELS = (16, 32)
CNN_KERNEL_SIZES = (7, 5)
CNN_POOL_KERNEL_SIZE = 2
NN_HIDDEN_LAYER_SIZES = (128, 64)
NN_ALPHA = 1e-4
NN_LEARNING_RATE_INIT = 1e-3
NN_BATCH_SIZE = 32
NN_MAX_ITER = 220
PREDICTION_THRESHOLD = 0.50
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Artifact ranges
UNIFORM_SHIFT_RANGE = (-0.12, 0.12)
SAMPLE_DISPLACEMENT_RANGE_MM = (-0.18, 0.18)
GONIOMETER_RADIUS_MM = 240.0

U_RANGE = (0.01, 0.06)
V_RANGE = (-0.02, 0.01)
W_RANGE = (0.002, 0.010)
SIZE_NM_RANGE = (8.0, 80.0)
MICROSTRAIN_RANGE = (0.0, 0.003)

FWHM_RANGE = (0.08, 0.45)
ETA_RANGE = (0.10, 0.70)

BACKGROUND_SCALE_RANGE = (0.05, 0.28)
HUMP_SCALE_RANGE = (0.02, 0.20)
NOISE_SCALE_RANGE = (0.002, 0.020)

# Split
VAL_FRACTION = 0.20


# -----------------------------
# Pattern simulation utilities
# -----------------------------
def normalize_0_100(y):
    y = np.asarray(y, dtype=float)
    y = y - y.min()
    return 100.0 * y / np.clip(y.max(), 1e-12, None)


def sample_displacement_shift(two_theta_deg, displacement_mm):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    d_relative_change = displacement_mm / GONIOMETER_RADIUS_MM * np.cos(theta_rad) ** 2
    return np.rad2deg(-d_relative_change * np.tan(theta_rad))


def instrumental_fwhm(two_theta_deg, u, v, w):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    tan_theta = np.tan(theta_rad)
    fwhm_sq = u * tan_theta**2 + v * tan_theta + w
    return np.sqrt(np.clip(fwhm_sq, 1e-4, None))


def size_fwhm(two_theta_deg, size_nm):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    wavelength_nm = WAVELENGTH_ANGSTROM / 10.0
    beta_rad = 0.9 * wavelength_nm / (size_nm * np.cos(theta_rad))
    return np.rad2deg(beta_rad)


def strain_fwhm(two_theta_deg, microstrain):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    beta_rad = 4.0 * microstrain * np.tan(theta_rad)
    return np.rad2deg(beta_rad)


def pseudo_voigt_profile(two_theta_grid, centers, fwhm, eta):
    dx = two_theta_grid[:, None] - centers[None, :]
    sigma = np.clip(fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0))), 1e-6, None)
    gamma = np.clip(fwhm / 2.0, 1e-6, None)
    gauss = np.exp(-0.5 * (dx / sigma[None, :]) ** 2)
    lorentz = (gamma[None, :] ** 2) / (dx**2 + gamma[None, :] ** 2)
    return (1.0 - eta) * gauss + eta * lorentz


def load_reference_sticks(cif_files):
    calc = XRDCalculator(wavelength=WAVELENGTH)
    refs_by_formula = {}

    for cif_file in cif_files:
        pattern = calc.get_pattern(Structure.from_file(cif_file), two_theta_range=(MIN_ANGLE, MAX_ANGLE))
        peak_pos = np.asarray(pattern.x, dtype=float)
        peak_int = np.asarray(pattern.y, dtype=float)

        keep = peak_int >= REFERENCE_INTENSITY_THRESHOLD
        peak_pos = peak_pos[keep]
        peak_int = normalize_0_100(peak_int[keep])

        formula = cif_file.stem.split("_", 1)[0]
        refs_by_formula.setdefault(formula, []).append({"phase": cif_file.stem, "peak_pos": peak_pos, "peak_int": peak_int})

    return refs_by_formula


def simulate_component_profile(two_theta_grid, base_pos, base_int, rng):
    if len(base_pos) == 0:
        return np.zeros_like(two_theta_grid)

    peak_int = base_int * np.exp(rng.normal(0.0, 0.25, size=len(base_int)))

    uniform_shift = rng.uniform(*UNIFORM_SHIFT_RANGE)
    displacement = rng.uniform(*SAMPLE_DISPLACEMENT_RANGE_MM)
    peak_pos = base_pos + uniform_shift + sample_displacement_shift(base_pos, displacement)

    keep = (peak_pos >= MIN_ANGLE - 1.0) & (peak_pos <= MAX_ANGLE + 1.0)
    peak_pos = peak_pos[keep]
    peak_int = peak_int[keep]
    if len(peak_pos) == 0:
        return np.zeros_like(two_theta_grid)

    u = rng.uniform(*U_RANGE)
    v = rng.uniform(*V_RANGE)
    w = rng.uniform(*W_RANGE)
    size_nm = rng.uniform(*SIZE_NM_RANGE)
    microstrain = rng.uniform(*MICROSTRAIN_RANGE)

    fwhm = np.sqrt(
        instrumental_fwhm(peak_pos, u, v, w) ** 2
        + size_fwhm(peak_pos, size_nm) ** 2
        + strain_fwhm(peak_pos, microstrain) ** 2
    )
    fwhm += rng.uniform(*FWHM_RANGE)
    eta = rng.uniform(*ETA_RANGE)

    profile = pseudo_voigt_profile(two_theta_grid, peak_pos, fwhm, eta) @ peak_int
    return profile / np.clip(profile.max(), 1e-12, None)


def simulate_multiphase_profile(two_theta_grid, formula_list, refs_by_formula, rng):
    components = []
    for formula in formula_list:
        ref = refs_by_formula[formula][rng.integers(0, len(refs_by_formula[formula]))]
        components.append(simulate_component_profile(two_theta_grid, ref["peak_pos"], ref["peak_int"], rng))

    weights = rng.dirichlet(np.ones(len(components)) * 1.5)
    peaks = np.zeros_like(two_theta_grid)
    for w, comp in zip(weights, components):
        peaks += w * comp
    peaks = normalize_0_100(peaks)

    x_cheb = 2.0 * (two_theta_grid - MIN_ANGLE) / (MAX_ANGLE - MIN_ANGLE) - 1.0
    coeffs = np.array([1.0, rng.uniform(-0.5, 0.5), rng.uniform(-0.4, 0.4), rng.uniform(-0.2, 0.2), rng.uniform(-0.1, 0.1)])
    background = np.polynomial.chebyshev.chebval(x_cheb, coeffs)
    background -= background.min()
    background /= np.clip(background.max(), 1e-12, None)
    background *= rng.uniform(*BACKGROUND_SCALE_RANGE) * peaks.max()

    center = rng.uniform(18.0, 35.0)
    width = rng.uniform(5.0, 12.0)
    hump = rng.uniform(*HUMP_SCALE_RANGE) * peaks.max() * np.exp(-0.5 * ((two_theta_grid - center) / width) ** 2)

    noise_sigma = rng.uniform(*NOISE_SCALE_RANGE) * peaks.max()
    noise = rng.normal(0.0, noise_sigma, size=len(two_theta_grid))

    y = peaks + background + hump + noise
    y -= y.min()
    return normalize_0_100(y)


def build_synthetic_multiphase_dataset(refs_by_formula, two_theta_grid, rng):
    formulas = sorted(refs_by_formula.keys())
    X = []
    y_labels = []

    for anchor in formulas:
        others = [f for f in formulas if f != anchor]
        for _ in range(SYNTH_SAMPLES_PER_FORMULA):
            n_components = int(
                rng.choice(
                    [1, 2, 3],
                    p=[
                        SINGLE_PHASE_FRACTION,
                        (1.0 - SINGLE_PHASE_FRACTION) * 0.65,
                        (1.0 - SINGLE_PHASE_FRACTION) * 0.35,
                    ],
                )
            )
            chosen_others = list(rng.choice(others, size=n_components - 1, replace=False))
            labels = sorted([anchor] + chosen_others)

            profile = simulate_multiphase_profile(two_theta_grid, labels, refs_by_formula, rng)
            X.append(profile)
            y_labels.append(labels)

    return np.asarray(X), y_labels


def preprocess_experimental_pattern(xy_file, two_theta_grid):
    data = np.loadtxt(xy_file)
    x = data[:, 0]
    y = data[:, 1]

    keep = (x >= MIN_ANGLE) & (x <= MAX_ANGLE)
    x = x[keep]
    y = y[keep]

    y_interp = np.interp(two_theta_grid, x, y)
    return normalize_0_100(y_interp)


# -----------------------------
# Multi-label NN utilities
# -----------------------------
def labels_from_filename(file_stem):
    return file_stem.split("_")


class SimpleConvNet(nn.Module):
    def __init__(self, n_outputs):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(1, CNN_CONV_CHANNELS[0], kernel_size=CNN_KERNEL_SIZES[0], padding=CNN_KERNEL_SIZES[0] // 2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=CNN_POOL_KERNEL_SIZE),
            nn.Conv1d(
                CNN_CONV_CHANNELS[0],
                CNN_CONV_CHANNELS[1],
                kernel_size=CNN_KERNEL_SIZES[1],
                padding=CNN_KERNEL_SIZES[1] // 2,
            ),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=CNN_POOL_KERNEL_SIZE),
        )

        downsample_factor = CNN_POOL_KERNEL_SIZE**2
        conv_output_points = NUM_POINTS // downsample_factor
        conv_output_size = CNN_CONV_CHANNELS[1] * conv_output_points

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(conv_output_size, NN_HIDDEN_LAYER_SIZES[0]),
            nn.ReLU(),
            nn.Linear(NN_HIDDEN_LAYER_SIZES[0], NN_HIDDEN_LAYER_SIZES[1]),
            nn.ReLU(),
            nn.Linear(NN_HIDDEN_LAYER_SIZES[1], n_outputs),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def build_simple_nn(n_outputs):
    return SimpleConvNet(n_outputs).to(DEVICE)


def fit_simple_nn(model, X_train, y_train):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
    y_train = y_train.astype(np.float32)

    train_dataset = TensorDataset(
        torch.from_numpy(X_train_scaled[:, None, :]),
        torch.from_numpy(y_train),
    )
    train_loader = DataLoader(train_dataset, batch_size=NN_BATCH_SIZE, shuffle=True)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=NN_LEARNING_RATE_INIT, weight_decay=NN_ALPHA)

    loss_curve = []
    model.train()
    for _ in range(NN_MAX_ITER):
        epoch_loss = 0.0
        n_seen = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            batch_size = X_batch.shape[0]
            epoch_loss += loss.item() * batch_size
            n_seen += batch_size

        loss_curve.append(epoch_loss / np.clip(n_seen, 1, None))

    return scaler, np.asarray(loss_curve, dtype=float)


def get_label_scores(model, scaler, X):
    X_scaled = scaler.transform(X).astype(np.float32)
    X_tensor = torch.from_numpy(X_scaled[:, None, :]).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)
        scores = torch.sigmoid(logits).cpu().numpy()

    return scores


def predict_binary_labels(scores, threshold=PREDICTION_THRESHOLD):
    return (scores >= threshold).astype(int)


def compute_micro_metrics(y_true_bin, y_pred_bin):
    precision_micro = precision_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    recall_micro = recall_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    f1_micro = f1_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    return precision_micro, recall_micro, f1_micro


def plot_loss_curve(train_loss_curve):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    epochs = np.arange(1, len(train_loss_curve) + 1)

    ax.plot(epochs, train_loss_curve, color="tab:blue", linewidth=2.2, label="Train")

    ax.set_xlabel("Epoch", fontsize=18, labelpad=8)
    ax.set_ylabel("Loss", fontsize=18, labelpad=10)
    ax.tick_params(axis="both", labelsize=16)
    ax.legend(fontsize=16, loc="upper right", framealpha=1)
    ax.grid(alpha=0.25)

    out_file = OUTPUT_DIR / "nn_loss_curve.png"
    plt.tight_layout()
    plt.savefig(out_file, dpi=200)
    plt.close(fig)
    print(f"Saved plot: {out_file}")


def plot_test_metric_summary(precision_micro, recall_micro, f1_micro):
    fig, ax = plt.subplots(figsize=(6.6, 4.5))

    x = np.array([0])
    width = 0.24
    ax.bar(x - width, [precision_micro], width=width, color="tab:blue", edgecolor="black", linewidth=1.0, label="Precision")
    ax.bar(x, [recall_micro], width=width, color="tab:green", edgecolor="black", linewidth=1.0, label="Recall")
    ax.bar(x + width, [f1_micro], width=width, color="tab:red", edgecolor="black", linewidth=1.0, label="F1-score")

    ax.set_xticks(x)
    ax.set_xticklabels(["CNN"], fontsize=16)
    ax.tick_params(axis="y", labelsize=16)
    ax.set_ylim(0.0, 1.05)
    ax.set_ylabel("Score", fontsize=18, labelpad=12)
    ax.legend(fontsize=16, loc="lower right", framealpha=1)
    ax.grid(axis="y", alpha=0.25)

    out_file = OUTPUT_DIR / "nn_test-metric_summary.png"
    plt.tight_layout()
    plt.savefig(out_file, dpi=200)
    plt.close(fig)
    print(f"Saved plot: {out_file}")


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(RANDOM_SEED)
    two_theta_grid = np.linspace(MIN_ANGLE, MAX_ANGLE, NUM_POINTS)

    refs_by_formula = load_reference_sticks(sorted(REFERENCE_DIR.glob("*.cif")))
    all_formulas = sorted(refs_by_formula.keys())

    X, y_label_lists = build_synthetic_multiphase_dataset(refs_by_formula, two_theta_grid, rng)

    mlb = MultiLabelBinarizer(classes=all_formulas)
    y_bin = mlb.fit_transform(y_label_lists)

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y_bin,
        test_size=VAL_FRACTION,
        random_state=RANDOM_SEED,
    )

    exp_files = sorted(EXPERIMENT_DIR.glob("*.xy"))
    X_test = np.asarray([preprocess_experimental_pattern(f, two_theta_grid) for f in exp_files])
    y_test_labels = [labels_from_filename(f.stem) for f in exp_files]
    y_test = mlb.transform(y_test_labels)

    print("\n=== CNN Phase-ID Demo (multi-phase) ===")
    print(f"Synthetic samples: {len(X)}")
    print(f"Unique formulas (labels): {len(all_formulas)}")
    print(f"Training samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    print(f"Experimental test patterns: {len(X_test)}")

    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)

    model = build_simple_nn(n_outputs=len(all_formulas))
    scaler, train_loss_curve = fit_simple_nn(model, X_train, y_train)

    val_scores = get_label_scores(model, scaler, X_val)
    y_pred_val = predict_binary_labels(val_scores, threshold=PREDICTION_THRESHOLD)
    val_precision_micro, val_recall_micro, val_f1_micro = compute_micro_metrics(y_val, y_pred_val)

    test_scores = get_label_scores(model, scaler, X_test)
    y_pred_test = predict_binary_labels(test_scores, threshold=PREDICTION_THRESHOLD)
    test_precision_micro, test_recall_micro, test_f1_micro = compute_micro_metrics(y_test, y_pred_test)

    print("\nNeural Net")
    print(f"  Conv layers: {CNN_CONV_CHANNELS} kernels={CNN_KERNEL_SIZES} pool={CNN_POOL_KERNEL_SIZE}")
    print(f"  Fixed hidden layers: {NN_HIDDEN_LAYER_SIZES}")
    print(f"  Output nodes (labels): {len(all_formulas)}")
    print(f"  Binary threshold: {PREDICTION_THRESHOLD:.2f}")
    print(f"  Validation precision (micro): {val_precision_micro:.3f}")
    print(f"  Validation recall (micro): {val_recall_micro:.3f}")
    print(f"  Validation F1 (micro): {val_f1_micro:.3f}")
    print(f"  Test precision (micro): {test_precision_micro:.3f}")
    print(f"  Test recall (micro): {test_recall_micro:.3f}")
    print(f"  Test F1 (micro): {test_f1_micro:.3f}")

    print("  Test predictions (binary outputs):")
    prediction_rows = []
    for i, exp_file in enumerate(exp_files):
        true_set = y_test_labels[i]
        pred_set = list(mlb.classes_[np.where(y_pred_test[i] == 1)[0]])

        true_binary = " ".join(map(str, y_test[i].astype(int).tolist()))
        pred_binary = " ".join(map(str, y_pred_test[i].astype(int).tolist()))
        inter = len(set(true_set) & set(pred_set))
        f1_pattern = 0.0 if (len(true_set) + len(pred_set)) == 0 else 2.0 * inter / (len(true_set) + len(pred_set))

        print(f"    {exp_file.stem:26s} -> [{pred_binary}] {pred_set}")
        prediction_rows.append(
            {
                "pattern": exp_file.stem,
                "true_labels": ";".join(true_set),
                "predicted_labels": ";".join(pred_set),
                "true_binary": true_binary,
                "predicted_binary": pred_binary,
                "n_predicted": len(pred_set),
                "f1_pattern": f1_pattern,
            }
        )

    if len(train_loss_curve) > 0:
        plot_loss_curve(train_loss_curve)
    plot_test_metric_summary(test_precision_micro, test_recall_micro, test_f1_micro)

    metrics_file = OUTPUT_DIR / "nn_metrics.csv"
    with open(metrics_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "model",
                "hidden_layers",
                "n_outputs",
                "threshold",
                "validation_precision_micro",
                "validation_recall_micro",
                "validation_f1_micro",
                "test_precision_micro",
                "test_recall_micro",
                "test_f1_micro",
            ],
        )
        writer.writeheader()
        writer.writerow(
            {
                "model": "CNN",
                "hidden_layers": str(NN_HIDDEN_LAYER_SIZES),
                "n_outputs": len(all_formulas),
                "threshold": PREDICTION_THRESHOLD,
                "validation_precision_micro": val_precision_micro,
                "validation_recall_micro": val_recall_micro,
                "validation_f1_micro": val_f1_micro,
                "test_precision_micro": test_precision_micro,
                "test_recall_micro": test_recall_micro,
                "test_f1_micro": test_f1_micro,
            }
        )

    pred_file = OUTPUT_DIR / "test_predictions_binary.csv"
    with open(pred_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "pattern",
                "true_labels",
                "predicted_labels",
                "true_binary",
                "predicted_binary",
                "n_predicted",
                "f1_pattern",
            ],
        )
        writer.writeheader()
        writer.writerows(prediction_rows)

    print(f"\nSaved metrics: {metrics_file}")
    print(f"Saved predictions: {pred_file}")


# 04c — CNNs: Multiphase

A 1D CNN is trained on synthetic mixtures to capture local peak-shape and neighborhood features.

## Option A / Option B
Option A: run short training (shown below).

Option B: if you provide `data/pretrained/cnn_multiphase.pt`, load it in your own inference workflow.

In [ ]:
PRETRAINED_PATH = "data/pretrained/cnn_multiphase.pt"
print("Found pretrained weights:", os.path.exists(PRETRAINED_PATH))

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
SYNTH_SAMPLES_PER_FORMULA = 8
NN_MAX_ITER = 20  # Reduced for demo speed
main()


## What To Observe
Compare the CNN metric summary against the dense-NN multiphase baseline.

In [ ]:
display(Image("outputs/dl/cnn_multiphase/nn_loss_curve.png"))
display(Image("outputs/dl/cnn_multiphase/nn_test-metric_summary.png"))

## Summary
- 1D CNNs encode local line-shape context directly.
- Short demo training is useful for workflow illustration, not final performance.
- Checkpoints are recommended for live sessions with tight time budgets.

## Next Steps
Continue to the next section below in this notebook.

## 04d - No Augmentation

In [ ]:
import os
from IPython.display import Image, display

# Inline tutorial script
from pathlib import Path
import csv

# For handling arrays
import numpy as np

# For plotting
import matplotlib.pyplot as plt

# To load structures and compute XRD stick patterns
from pymatgen.core import Structure
from pymatgen.analysis.diffraction.xrd import XRDCalculator

# Neural-network model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.metrics import precision_score, recall_score, f1_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


# Input/output
EXPERIMENT_DIR = Path("data/exp_patterns/multi_phase")
REFERENCE_DIR = Path("data/reference_structures")
OUTPUT_DIR = Path("outputs/dl/cnn_no_augmentation")

# Pattern settings
MIN_ANGLE = 10.0
MAX_ANGLE = 80.0
NUM_POINTS = 1400
WAVELENGTH = "CuKa"
WAVELENGTH_ANGSTROM = 1.5406
REFERENCE_INTENSITY_THRESHOLD = 1.0

# Synthetic-data settings
SYNTH_SAMPLES_PER_FORMULA = 60
RANDOM_SEED = 42
SINGLE_PHASE_FRACTION = 0.15

# Neural-net settings (fixed architecture; no tuning)
CNN_CONV_CHANNELS = (16, 32)
CNN_KERNEL_SIZES = (7, 5)
CNN_POOL_KERNEL_SIZE = 2
NN_HIDDEN_LAYER_SIZES = (128, 64)
NN_ALPHA = 1e-4
NN_LEARNING_RATE_INIT = 1e-3
NN_BATCH_SIZE = 32
NN_MAX_ITER = 220
PREDICTION_THRESHOLD = 0.50
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Idealized synthetic-pattern settings (no artifacts)
IDEAL_FWHM = 0.18
IDEAL_ETA = 0.0

# Split
VAL_FRACTION = 0.20


# -----------------------------
# Pattern simulation utilities
# -----------------------------
def normalize_0_100(y):
    y = np.asarray(y, dtype=float)
    y = y - y.min()
    return 100.0 * y / np.clip(y.max(), 1e-12, None)


def pseudo_voigt_profile(two_theta_grid, centers, fwhm, eta):
    dx = two_theta_grid[:, None] - centers[None, :]
    sigma = np.clip(fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0))), 1e-6, None)
    gamma = np.clip(fwhm / 2.0, 1e-6, None)
    gauss = np.exp(-0.5 * (dx / sigma[None, :]) ** 2)
    lorentz = (gamma[None, :] ** 2) / (dx**2 + gamma[None, :] ** 2)
    return (1.0 - eta) * gauss + eta * lorentz


def load_reference_sticks(cif_files):
    calc = XRDCalculator(wavelength=WAVELENGTH)
    refs_by_formula = {}

    for cif_file in cif_files:
        pattern = calc.get_pattern(Structure.from_file(cif_file), two_theta_range=(MIN_ANGLE, MAX_ANGLE))
        peak_pos = np.asarray(pattern.x, dtype=float)
        peak_int = np.asarray(pattern.y, dtype=float)

        keep = peak_int >= REFERENCE_INTENSITY_THRESHOLD
        peak_pos = peak_pos[keep]
        peak_int = normalize_0_100(peak_int[keep])

        formula = cif_file.stem.split("_", 1)[0]
        refs_by_formula.setdefault(formula, []).append({"phase": cif_file.stem, "peak_pos": peak_pos, "peak_int": peak_int})

    return refs_by_formula


def simulate_component_profile(two_theta_grid, base_pos, base_int, rng):
    if len(base_pos) == 0:
        return np.zeros_like(two_theta_grid)

    keep = (base_pos >= MIN_ANGLE - 1.0) & (base_pos <= MAX_ANGLE + 1.0)
    peak_pos = base_pos[keep]
    peak_int = base_int[keep]
    if len(peak_pos) == 0:
        return np.zeros_like(two_theta_grid)

    fwhm = np.full(len(peak_pos), IDEAL_FWHM, dtype=float)
    profile = pseudo_voigt_profile(two_theta_grid, peak_pos, fwhm, IDEAL_ETA) @ peak_int
    return profile / np.clip(profile.max(), 1e-12, None)


def simulate_multiphase_profile(two_theta_grid, formula_list, refs_by_formula, rng):
    components = []
    for formula in formula_list:
        ref = refs_by_formula[formula][rng.integers(0, len(refs_by_formula[formula]))]
        components.append(simulate_component_profile(two_theta_grid, ref["peak_pos"], ref["peak_int"], rng))

    weights = rng.dirichlet(np.ones(len(components)) * 1.5)
    peaks = np.zeros_like(two_theta_grid)
    for w, comp in zip(weights, components):
        peaks += w * comp

    return normalize_0_100(peaks)


def build_synthetic_multiphase_dataset(refs_by_formula, two_theta_grid, rng):
    formulas = sorted(refs_by_formula.keys())
    X = []
    y_labels = []

    for anchor in formulas:
        others = [f for f in formulas if f != anchor]
        for _ in range(SYNTH_SAMPLES_PER_FORMULA):
            n_components = int(
                rng.choice(
                    [1, 2, 3],
                    p=[
                        SINGLE_PHASE_FRACTION,
                        (1.0 - SINGLE_PHASE_FRACTION) * 0.65,
                        (1.0 - SINGLE_PHASE_FRACTION) * 0.35,
                    ],
                )
            )
            chosen_others = list(rng.choice(others, size=n_components - 1, replace=False))
            labels = sorted([anchor] + chosen_others)

            profile = simulate_multiphase_profile(two_theta_grid, labels, refs_by_formula, rng)
            X.append(profile)
            y_labels.append(labels)

    return np.asarray(X), y_labels


def preprocess_experimental_pattern(xy_file, two_theta_grid):
    data = np.loadtxt(xy_file)
    x = data[:, 0]
    y = data[:, 1]

    keep = (x >= MIN_ANGLE) & (x <= MAX_ANGLE)
    x = x[keep]
    y = y[keep]

    y_interp = np.interp(two_theta_grid, x, y)
    return normalize_0_100(y_interp)


# -----------------------------
# Multi-label NN utilities
# -----------------------------
def labels_from_filename(file_stem):
    return file_stem.split("_")


class SimpleConvNet(nn.Module):
    def __init__(self, n_outputs):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(1, CNN_CONV_CHANNELS[0], kernel_size=CNN_KERNEL_SIZES[0], padding=CNN_KERNEL_SIZES[0] // 2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=CNN_POOL_KERNEL_SIZE),
            nn.Conv1d(
                CNN_CONV_CHANNELS[0],
                CNN_CONV_CHANNELS[1],
                kernel_size=CNN_KERNEL_SIZES[1],
                padding=CNN_KERNEL_SIZES[1] // 2,
            ),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=CNN_POOL_KERNEL_SIZE),
        )

        downsample_factor = CNN_POOL_KERNEL_SIZE**2
        conv_output_points = NUM_POINTS // downsample_factor
        conv_output_size = CNN_CONV_CHANNELS[1] * conv_output_points

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(conv_output_size, NN_HIDDEN_LAYER_SIZES[0]),
            nn.ReLU(),
            nn.Linear(NN_HIDDEN_LAYER_SIZES[0], NN_HIDDEN_LAYER_SIZES[1]),
            nn.ReLU(),
            nn.Linear(NN_HIDDEN_LAYER_SIZES[1], n_outputs),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def build_simple_nn(n_outputs):
    return SimpleConvNet(n_outputs).to(DEVICE)


def fit_simple_nn(model, X_train, y_train):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
    y_train = y_train.astype(np.float32)

    train_dataset = TensorDataset(
        torch.from_numpy(X_train_scaled[:, None, :]),
        torch.from_numpy(y_train),
    )
    train_loader = DataLoader(train_dataset, batch_size=NN_BATCH_SIZE, shuffle=True)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=NN_LEARNING_RATE_INIT, weight_decay=NN_ALPHA)

    loss_curve = []
    model.train()
    for _ in range(NN_MAX_ITER):
        epoch_loss = 0.0
        n_seen = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            batch_size = X_batch.shape[0]
            epoch_loss += loss.item() * batch_size
            n_seen += batch_size

        loss_curve.append(epoch_loss / np.clip(n_seen, 1, None))

    return scaler, np.asarray(loss_curve, dtype=float)


def get_label_scores(model, scaler, X):
    X_scaled = scaler.transform(X).astype(np.float32)
    X_tensor = torch.from_numpy(X_scaled[:, None, :]).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)
        scores = torch.sigmoid(logits).cpu().numpy()

    return scores


def predict_binary_labels(scores, threshold=PREDICTION_THRESHOLD):
    return (scores >= threshold).astype(int)


def compute_micro_metrics(y_true_bin, y_pred_bin):
    precision_micro = precision_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    recall_micro = recall_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    f1_micro = f1_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    return precision_micro, recall_micro, f1_micro


def plot_loss_curve(train_loss_curve):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    epochs = np.arange(1, len(train_loss_curve) + 1)

    ax.plot(epochs, train_loss_curve, color="tab:blue", linewidth=2.2, label="Train")

    ax.set_xlabel("Epoch", fontsize=18, labelpad=8)
    ax.set_ylabel("Loss", fontsize=18, labelpad=10)
    ax.tick_params(axis="both", labelsize=16)
    ax.legend(fontsize=16, loc="upper right", framealpha=1)
    ax.grid(alpha=0.25)

    out_file = OUTPUT_DIR / "nn_loss_curve.png"
    plt.tight_layout()
    plt.savefig(out_file, dpi=200)
    plt.close(fig)
    print(f"Saved plot: {out_file}")


def plot_test_metric_summary(precision_micro, recall_micro, f1_micro):
    fig, ax = plt.subplots(figsize=(6.6, 4.5))

    x = np.array([0])
    width = 0.24
    ax.bar(x - width, [precision_micro], width=width, color="tab:blue", edgecolor="black", linewidth=1.0, label="Precision")
    ax.bar(x, [recall_micro], width=width, color="tab:green", edgecolor="black", linewidth=1.0, label="Recall")
    ax.bar(x + width, [f1_micro], width=width, color="tab:red", edgecolor="black", linewidth=1.0, label="F1-score")

    ax.set_xticks(x)
    ax.set_xticklabels(["CNN"], fontsize=16)
    ax.tick_params(axis="y", labelsize=16)
    ax.set_ylim(0.0, 1.05)
    ax.set_ylabel("Score", fontsize=18, labelpad=12)
    ax.legend(fontsize=16, loc="lower right", framealpha=1)
    ax.grid(axis="y", alpha=0.25)

    out_file = OUTPUT_DIR / "nn_test-metric_summary.png"
    plt.tight_layout()
    plt.savefig(out_file, dpi=200)
    plt.close(fig)
    print(f"Saved plot: {out_file}")


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(RANDOM_SEED)
    two_theta_grid = np.linspace(MIN_ANGLE, MAX_ANGLE, NUM_POINTS)

    refs_by_formula = load_reference_sticks(sorted(REFERENCE_DIR.glob("*.cif")))
    all_formulas = sorted(refs_by_formula.keys())

    X, y_label_lists = build_synthetic_multiphase_dataset(refs_by_formula, two_theta_grid, rng)

    mlb = MultiLabelBinarizer(classes=all_formulas)
    y_bin = mlb.fit_transform(y_label_lists)

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y_bin,
        test_size=VAL_FRACTION,
        random_state=RANDOM_SEED,
    )

    exp_files = sorted(EXPERIMENT_DIR.glob("*.xy"))
    X_test = np.asarray([preprocess_experimental_pattern(f, two_theta_grid) for f in exp_files])
    y_test_labels = [labels_from_filename(f.stem) for f in exp_files]
    y_test = mlb.transform(y_test_labels)

    print("\n=== CNN Phase-ID Demo (no augmentation; idealized synthetic data) ===")
    print(f"Synthetic samples: {len(X)}")
    print(f"Unique formulas (labels): {len(all_formulas)}")
    print(f"Training samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    print(f"Experimental test patterns: {len(X_test)}")

    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)

    model = build_simple_nn(n_outputs=len(all_formulas))
    scaler, train_loss_curve = fit_simple_nn(model, X_train, y_train)

    val_scores = get_label_scores(model, scaler, X_val)
    y_pred_val = predict_binary_labels(val_scores, threshold=PREDICTION_THRESHOLD)
    val_precision_micro, val_recall_micro, val_f1_micro = compute_micro_metrics(y_val, y_pred_val)

    test_scores = get_label_scores(model, scaler, X_test)
    y_pred_test = predict_binary_labels(test_scores, threshold=PREDICTION_THRESHOLD)
    test_precision_micro, test_recall_micro, test_f1_micro = compute_micro_metrics(y_test, y_pred_test)

    print("\nCNN (No Augmentation)")
    print(f"  Conv layers: {CNN_CONV_CHANNELS} kernels={CNN_KERNEL_SIZES} pool={CNN_POOL_KERNEL_SIZE}")
    print(f"  Fixed hidden layers: {NN_HIDDEN_LAYER_SIZES}")
    print(f"  Output nodes (labels): {len(all_formulas)}")
    print(f"  Binary threshold: {PREDICTION_THRESHOLD:.2f}")
    print(f"  Validation precision (micro): {val_precision_micro:.3f}")
    print(f"  Validation recall (micro): {val_recall_micro:.3f}")
    print(f"  Validation F1 (micro): {val_f1_micro:.3f}")
    print(f"  Test precision (micro): {test_precision_micro:.3f}")
    print(f"  Test recall (micro): {test_recall_micro:.3f}")
    print(f"  Test F1 (micro): {test_f1_micro:.3f}")

    print("  Test predictions (binary outputs):")
    prediction_rows = []
    for i, exp_file in enumerate(exp_files):
        true_set = y_test_labels[i]
        pred_set = list(mlb.classes_[np.where(y_pred_test[i] == 1)[0]])

        true_binary = " ".join(map(str, y_test[i].astype(int).tolist()))
        pred_binary = " ".join(map(str, y_pred_test[i].astype(int).tolist()))
        inter = len(set(true_set) & set(pred_set))
        f1_pattern = 0.0 if (len(true_set) + len(pred_set)) == 0 else 2.0 * inter / (len(true_set) + len(pred_set))

        print(f"    {exp_file.stem:26s} -> [{pred_binary}] {pred_set}")
        prediction_rows.append(
            {
                "pattern": exp_file.stem,
                "true_labels": ";".join(true_set),
                "predicted_labels": ";".join(pred_set),
                "true_binary": true_binary,
                "predicted_binary": pred_binary,
                "n_predicted": len(pred_set),
                "f1_pattern": f1_pattern,
            }
        )

    if len(train_loss_curve) > 0:
        plot_loss_curve(train_loss_curve)
    plot_test_metric_summary(test_precision_micro, test_recall_micro, test_f1_micro)

    metrics_file = OUTPUT_DIR / "nn_metrics.csv"
    with open(metrics_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "model",
                "hidden_layers",
                "n_outputs",
                "threshold",
                "validation_precision_micro",
                "validation_recall_micro",
                "validation_f1_micro",
                "test_precision_micro",
                "test_recall_micro",
                "test_f1_micro",
            ],
        )
        writer.writeheader()
        writer.writerow(
            {
                "model": "CNN",
                "hidden_layers": str(NN_HIDDEN_LAYER_SIZES),
                "n_outputs": len(all_formulas),
                "threshold": PREDICTION_THRESHOLD,
                "validation_precision_micro": val_precision_micro,
                "validation_recall_micro": val_recall_micro,
                "validation_f1_micro": val_f1_micro,
                "test_precision_micro": test_precision_micro,
                "test_recall_micro": test_recall_micro,
                "test_f1_micro": test_f1_micro,
            }
        )

    pred_file = OUTPUT_DIR / "test_predictions_binary.csv"
    with open(pred_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "pattern",
                "true_labels",
                "predicted_labels",
                "true_binary",
                "predicted_binary",
                "n_predicted",
                "f1_pattern",
            ],
        )
        writer.writeheader()
        writer.writerows(prediction_rows)

    print(f"\nSaved metrics: {metrics_file}")
    print(f"Saved predictions: {pred_file}")


# 04d — Ablation: No Augmentation

This ablation trains a CNN on idealized synthetic data without the broader artifact model.

## Option A / Option B
Option A: run short no-augmentation training.

Option B: use a saved checkpoint (if provided) from `data/pretrained/`.

In [ ]:
PRETRAINED_PATH = "data/pretrained/cnn_no_augmentation.pt"
print("Found pretrained weights:", os.path.exists(PRETRAINED_PATH))

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
SYNTH_SAMPLES_PER_FORMULA = 8
NN_MAX_ITER = 20  # Reduced for demo speed
main()


## What To Observe
Use this as a baseline to compare against artifact-aware augmentation variants.

In [ ]:
display(Image("outputs/dl/cnn_no_augmentation/nn_loss_curve.png"))
display(Image("outputs/dl/cnn_no_augmentation/nn_test-metric_summary.png"))

## Summary
- No-augmentation training is a useful controlled baseline.
- Generalization typically drops when synthetic variability is too narrow.
- Ablations help quantify which augmentations matter most.

## Next Steps
Continue to the next section below in this notebook.

## 04e - Random Shifts

In [ ]:
import os
from IPython.display import Image, display

# Inline tutorial script
from pathlib import Path
import csv

# For handling arrays
import numpy as np

# For plotting
import matplotlib.pyplot as plt

# To load structures and compute XRD stick patterns
from pymatgen.core import Structure
from pymatgen.analysis.diffraction.xrd import XRDCalculator

# Neural-network model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.metrics import precision_score, recall_score, f1_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


# Input/output
EXPERIMENT_DIR = Path("data/exp_patterns/multi_phase")
REFERENCE_DIR = Path("data/reference_structures")
OUTPUT_DIR = Path("outputs/dl/cnn_random_shifts")

# Pattern settings
MIN_ANGLE = 10.0
MAX_ANGLE = 80.0
NUM_POINTS = 1400
WAVELENGTH = "CuKa"
WAVELENGTH_ANGSTROM = 1.5406
REFERENCE_INTENSITY_THRESHOLD = 1.0

# Synthetic-data settings
SYNTH_SAMPLES_PER_FORMULA = 60
RANDOM_SEED = 42
SINGLE_PHASE_FRACTION = 0.15

# Neural-net settings (fixed architecture; no tuning)
CNN_CONV_CHANNELS = (16, 32)
CNN_KERNEL_SIZES = (7, 5)
CNN_POOL_KERNEL_SIZE = 2
NN_HIDDEN_LAYER_SIZES = (128, 64)
NN_ALPHA = 1e-4
NN_LEARNING_RATE_INIT = 1e-3
NN_BATCH_SIZE = 32
NN_MAX_ITER = 220
PREDICTION_THRESHOLD = 0.50
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Synthetic-pattern settings (random peak shifts only; no artifact model)
IDEAL_FWHM = 0.18
IDEAL_ETA = 0.0
MAX_PEAK_SHIFT_MAGNITUDE = 0.5

# Split
VAL_FRACTION = 0.20


# -----------------------------
# Pattern simulation utilities
# -----------------------------
def normalize_0_100(y):
    y = np.asarray(y, dtype=float)
    y = y - y.min()
    return 100.0 * y / np.clip(y.max(), 1e-12, None)


def pseudo_voigt_profile(two_theta_grid, centers, fwhm, eta):
    dx = two_theta_grid[:, None] - centers[None, :]
    sigma = np.clip(fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0))), 1e-6, None)
    gamma = np.clip(fwhm / 2.0, 1e-6, None)
    gauss = np.exp(-0.5 * (dx / sigma[None, :]) ** 2)
    lorentz = (gamma[None, :] ** 2) / (dx**2 + gamma[None, :] ** 2)
    return (1.0 - eta) * gauss + eta * lorentz


def load_reference_sticks(cif_files):
    calc = XRDCalculator(wavelength=WAVELENGTH)
    refs_by_formula = {}

    for cif_file in cif_files:
        pattern = calc.get_pattern(Structure.from_file(cif_file), two_theta_range=(MIN_ANGLE, MAX_ANGLE))
        peak_pos = np.asarray(pattern.x, dtype=float)
        peak_int = np.asarray(pattern.y, dtype=float)

        keep = peak_int >= REFERENCE_INTENSITY_THRESHOLD
        peak_pos = peak_pos[keep]
        peak_int = normalize_0_100(peak_int[keep])

        formula = cif_file.stem.split("_", 1)[0]
        refs_by_formula.setdefault(formula, []).append({"phase": cif_file.stem, "peak_pos": peak_pos, "peak_int": peak_int})

    return refs_by_formula


def simulate_component_profile(two_theta_grid, base_pos, base_int, rng):
    if len(base_pos) == 0:
        return np.zeros_like(two_theta_grid)

    peak_pos = base_pos + rng.uniform(-MAX_PEAK_SHIFT_MAGNITUDE, MAX_PEAK_SHIFT_MAGNITUDE, size=len(base_pos))

    keep = (peak_pos >= MIN_ANGLE - 1.0) & (peak_pos <= MAX_ANGLE + 1.0)
    peak_pos = peak_pos[keep]
    peak_int = base_int[keep]
    if len(peak_pos) == 0:
        return np.zeros_like(two_theta_grid)

    fwhm = np.full(len(peak_pos), IDEAL_FWHM, dtype=float)
    profile = pseudo_voigt_profile(two_theta_grid, peak_pos, fwhm, IDEAL_ETA) @ peak_int
    return profile / np.clip(profile.max(), 1e-12, None)


def simulate_multiphase_profile(two_theta_grid, formula_list, refs_by_formula, rng):
    components = []
    for formula in formula_list:
        ref = refs_by_formula[formula][rng.integers(0, len(refs_by_formula[formula]))]
        components.append(simulate_component_profile(two_theta_grid, ref["peak_pos"], ref["peak_int"], rng))

    weights = rng.dirichlet(np.ones(len(components)) * 1.5)
    peaks = np.zeros_like(two_theta_grid)
    for w, comp in zip(weights, components):
        peaks += w * comp

    return normalize_0_100(peaks)


def build_synthetic_multiphase_dataset(refs_by_formula, two_theta_grid, rng):
    formulas = sorted(refs_by_formula.keys())
    X = []
    y_labels = []

    for anchor in formulas:
        others = [f for f in formulas if f != anchor]
        for _ in range(SYNTH_SAMPLES_PER_FORMULA):
            n_components = int(
                rng.choice(
                    [1, 2, 3],
                    p=[
                        SINGLE_PHASE_FRACTION,
                        (1.0 - SINGLE_PHASE_FRACTION) * 0.65,
                        (1.0 - SINGLE_PHASE_FRACTION) * 0.35,
                    ],
                )
            )
            chosen_others = list(rng.choice(others, size=n_components - 1, replace=False))
            labels = sorted([anchor] + chosen_others)

            profile = simulate_multiphase_profile(two_theta_grid, labels, refs_by_formula, rng)
            X.append(profile)
            y_labels.append(labels)

    return np.asarray(X), y_labels


def preprocess_experimental_pattern(xy_file, two_theta_grid):
    data = np.loadtxt(xy_file)
    x = data[:, 0]
    y = data[:, 1]

    keep = (x >= MIN_ANGLE) & (x <= MAX_ANGLE)
    x = x[keep]
    y = y[keep]

    y_interp = np.interp(two_theta_grid, x, y)
    return normalize_0_100(y_interp)


# -----------------------------
# Multi-label NN utilities
# -----------------------------
def labels_from_filename(file_stem):
    return file_stem.split("_")


class SimpleConvNet(nn.Module):
    def __init__(self, n_outputs):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(1, CNN_CONV_CHANNELS[0], kernel_size=CNN_KERNEL_SIZES[0], padding=CNN_KERNEL_SIZES[0] // 2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=CNN_POOL_KERNEL_SIZE),
            nn.Conv1d(
                CNN_CONV_CHANNELS[0],
                CNN_CONV_CHANNELS[1],
                kernel_size=CNN_KERNEL_SIZES[1],
                padding=CNN_KERNEL_SIZES[1] // 2,
            ),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=CNN_POOL_KERNEL_SIZE),
        )

        downsample_factor = CNN_POOL_KERNEL_SIZE**2
        conv_output_points = NUM_POINTS // downsample_factor
        conv_output_size = CNN_CONV_CHANNELS[1] * conv_output_points

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(conv_output_size, NN_HIDDEN_LAYER_SIZES[0]),
            nn.ReLU(),
            nn.Linear(NN_HIDDEN_LAYER_SIZES[0], NN_HIDDEN_LAYER_SIZES[1]),
            nn.ReLU(),
            nn.Linear(NN_HIDDEN_LAYER_SIZES[1], n_outputs),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def build_simple_nn(n_outputs):
    return SimpleConvNet(n_outputs).to(DEVICE)


def fit_simple_nn(model, X_train, y_train):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
    y_train = y_train.astype(np.float32)

    train_dataset = TensorDataset(
        torch.from_numpy(X_train_scaled[:, None, :]),
        torch.from_numpy(y_train),
    )
    train_loader = DataLoader(train_dataset, batch_size=NN_BATCH_SIZE, shuffle=True)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=NN_LEARNING_RATE_INIT, weight_decay=NN_ALPHA)

    loss_curve = []
    model.train()
    for _ in range(NN_MAX_ITER):
        epoch_loss = 0.0
        n_seen = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            batch_size = X_batch.shape[0]
            epoch_loss += loss.item() * batch_size
            n_seen += batch_size

        loss_curve.append(epoch_loss / np.clip(n_seen, 1, None))

    return scaler, np.asarray(loss_curve, dtype=float)


def get_label_scores(model, scaler, X):
    X_scaled = scaler.transform(X).astype(np.float32)
    X_tensor = torch.from_numpy(X_scaled[:, None, :]).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)
        scores = torch.sigmoid(logits).cpu().numpy()

    return scores


def predict_binary_labels(scores, threshold=PREDICTION_THRESHOLD):
    return (scores >= threshold).astype(int)


def compute_micro_metrics(y_true_bin, y_pred_bin):
    precision_micro = precision_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    recall_micro = recall_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    f1_micro = f1_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    return precision_micro, recall_micro, f1_micro


def plot_loss_curve(train_loss_curve):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    epochs = np.arange(1, len(train_loss_curve) + 1)

    ax.plot(epochs, train_loss_curve, color="tab:blue", linewidth=2.2, label="Train")

    ax.set_xlabel("Epoch", fontsize=18, labelpad=8)
    ax.set_ylabel("Loss", fontsize=18, labelpad=10)
    ax.tick_params(axis="both", labelsize=16)
    ax.legend(fontsize=16, loc="upper right", framealpha=1)
    ax.grid(alpha=0.25)

    out_file = OUTPUT_DIR / "nn_loss_curve.png"
    plt.tight_layout()
    plt.savefig(out_file, dpi=200)
    plt.close(fig)
    print(f"Saved plot: {out_file}")


def plot_test_metric_summary(precision_micro, recall_micro, f1_micro):
    fig, ax = plt.subplots(figsize=(6.6, 4.5))

    x = np.array([0])
    width = 0.24
    ax.bar(x - width, [precision_micro], width=width, color="tab:blue", edgecolor="black", linewidth=1.0, label="Precision")
    ax.bar(x, [recall_micro], width=width, color="tab:green", edgecolor="black", linewidth=1.0, label="Recall")
    ax.bar(x + width, [f1_micro], width=width, color="tab:red", edgecolor="black", linewidth=1.0, label="F1-score")

    ax.set_xticks(x)
    ax.set_xticklabels(["CNN"], fontsize=16)
    ax.tick_params(axis="y", labelsize=16)
    ax.set_ylim(0.0, 1.05)
    ax.set_ylabel("Score", fontsize=18, labelpad=12)
    ax.legend(fontsize=16, loc="lower right", framealpha=1)
    ax.grid(axis="y", alpha=0.25)

    out_file = OUTPUT_DIR / "nn_test-metric_summary.png"
    plt.tight_layout()
    plt.savefig(out_file, dpi=200)
    plt.close(fig)
    print(f"Saved plot: {out_file}")


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(RANDOM_SEED)
    two_theta_grid = np.linspace(MIN_ANGLE, MAX_ANGLE, NUM_POINTS)

    refs_by_formula = load_reference_sticks(sorted(REFERENCE_DIR.glob("*.cif")))
    all_formulas = sorted(refs_by_formula.keys())

    X, y_label_lists = build_synthetic_multiphase_dataset(refs_by_formula, two_theta_grid, rng)

    mlb = MultiLabelBinarizer(classes=all_formulas)
    y_bin = mlb.fit_transform(y_label_lists)

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y_bin,
        test_size=VAL_FRACTION,
        random_state=RANDOM_SEED,
    )

    exp_files = sorted(EXPERIMENT_DIR.glob("*.xy"))
    X_test = np.asarray([preprocess_experimental_pattern(f, two_theta_grid) for f in exp_files])
    y_test_labels = [labels_from_filename(f.stem) for f in exp_files]
    y_test = mlb.transform(y_test_labels)

    print("\n=== CNN Phase-ID Demo (random peak shifts; no artifact model) ===")
    print(f"Synthetic samples: {len(X)}")
    print(f"Unique formulas (labels): {len(all_formulas)}")
    print(f"Training samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    print(f"Experimental test patterns: {len(X_test)}")
    print(f"Max peak shift magnitude (deg 2theta): {MAX_PEAK_SHIFT_MAGNITUDE:.2f}")

    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)

    model = build_simple_nn(n_outputs=len(all_formulas))
    scaler, train_loss_curve = fit_simple_nn(model, X_train, y_train)

    val_scores = get_label_scores(model, scaler, X_val)
    y_pred_val = predict_binary_labels(val_scores, threshold=PREDICTION_THRESHOLD)
    val_precision_micro, val_recall_micro, val_f1_micro = compute_micro_metrics(y_val, y_pred_val)

    test_scores = get_label_scores(model, scaler, X_test)
    y_pred_test = predict_binary_labels(test_scores, threshold=PREDICTION_THRESHOLD)
    test_precision_micro, test_recall_micro, test_f1_micro = compute_micro_metrics(y_test, y_pred_test)

    print("\nCNN (Random Shift Augmentation)")
    print(f"  Conv layers: {CNN_CONV_CHANNELS} kernels={CNN_KERNEL_SIZES} pool={CNN_POOL_KERNEL_SIZE}")
    print(f"  Fixed hidden layers: {NN_HIDDEN_LAYER_SIZES}")
    print(f"  Output nodes (labels): {len(all_formulas)}")
    print(f"  Binary threshold: {PREDICTION_THRESHOLD:.2f}")
    print(f"  Validation precision (micro): {val_precision_micro:.3f}")
    print(f"  Validation recall (micro): {val_recall_micro:.3f}")
    print(f"  Validation F1 (micro): {val_f1_micro:.3f}")
    print(f"  Test precision (micro): {test_precision_micro:.3f}")
    print(f"  Test recall (micro): {test_recall_micro:.3f}")
    print(f"  Test F1 (micro): {test_f1_micro:.3f}")

    print("  Test predictions (binary outputs):")
    prediction_rows = []
    for i, exp_file in enumerate(exp_files):
        true_set = y_test_labels[i]
        pred_set = list(mlb.classes_[np.where(y_pred_test[i] == 1)[0]])

        true_binary = " ".join(map(str, y_test[i].astype(int).tolist()))
        pred_binary = " ".join(map(str, y_pred_test[i].astype(int).tolist()))
        inter = len(set(true_set) & set(pred_set))
        f1_pattern = 0.0 if (len(true_set) + len(pred_set)) == 0 else 2.0 * inter / (len(true_set) + len(pred_set))

        print(f"    {exp_file.stem:26s} -> [{pred_binary}] {pred_set}")
        prediction_rows.append(
            {
                "pattern": exp_file.stem,
                "true_labels": ";".join(true_set),
                "predicted_labels": ";".join(pred_set),
                "true_binary": true_binary,
                "predicted_binary": pred_binary,
                "n_predicted": len(pred_set),
                "f1_pattern": f1_pattern,
            }
        )

    if len(train_loss_curve) > 0:
        plot_loss_curve(train_loss_curve)
    plot_test_metric_summary(test_precision_micro, test_recall_micro, test_f1_micro)

    metrics_file = OUTPUT_DIR / "nn_metrics.csv"
    with open(metrics_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "model",
                "hidden_layers",
                "n_outputs",
                "threshold",
                "validation_precision_micro",
                "validation_recall_micro",
                "validation_f1_micro",
                "test_precision_micro",
                "test_recall_micro",
                "test_f1_micro",
            ],
        )
        writer.writeheader()
        writer.writerow(
            {
                "model": "CNN",
                "hidden_layers": str(NN_HIDDEN_LAYER_SIZES),
                "n_outputs": len(all_formulas),
                "threshold": PREDICTION_THRESHOLD,
                "validation_precision_micro": val_precision_micro,
                "validation_recall_micro": val_recall_micro,
                "validation_f1_micro": val_f1_micro,
                "test_precision_micro": test_precision_micro,
                "test_recall_micro": test_recall_micro,
                "test_f1_micro": test_f1_micro,
            }
        )

    pred_file = OUTPUT_DIR / "test_predictions_binary.csv"
    with open(pred_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "pattern",
                "true_labels",
                "predicted_labels",
                "true_binary",
                "predicted_binary",
                "n_predicted",
                "f1_pattern",
            ],
        )
        writer.writeheader()
        writer.writerows(prediction_rows)

    print(f"\nSaved metrics: {metrics_file}")
    print(f"Saved predictions: {pred_file}")


# 04e — Ablation: Random Shifts

This variant keeps idealized patterns but injects random peak-position shifts as a lightweight augmentation.

## Option A / Option B
Option A: run short random-shift training.

Option B: use a saved checkpoint from `data/pretrained/` if you have one.

In [ ]:
PRETRAINED_PATH = "data/pretrained/cnn_random_shifts.pt"
print("Found pretrained weights:", os.path.exists(PRETRAINED_PATH))

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
SYNTH_SAMPLES_PER_FORMULA = 8
NN_MAX_ITER = 20  # Reduced for demo speed
main()


## What To Observe
Compare this model against both no-augmentation and full-artifact CNN training.

In [ ]:
display(Image("outputs/dl/cnn_random_shifts/nn_loss_curve.png"))
display(Image("outputs/dl/cnn_random_shifts/nn_test-metric_summary.png"))

## Summary
- Random shifts target one important nuisance factor: peak-position drift.
- Single-factor augmentation usually helps less than full artifact simulation.
- Ablation studies clarify augmentation ROI.

## Next Steps
Continue to the next section below in this notebook.

## 04f - Mixture of Experts

In [ ]:
import os
from IPython.display import Image, display

# Inline tutorial script
from pathlib import Path
import csv

# For handling arrays
import numpy as np

# For plotting
import matplotlib.pyplot as plt

# To load structures and compute XRD stick patterns
from pymatgen.core import Structure
from pymatgen.analysis.diffraction.xrd import XRDCalculator

# Neural-network model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.metrics import precision_score, recall_score, f1_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


# Input/output
EXPERIMENT_DIR = Path("data/exp_patterns/multi_phase")
REFERENCE_DIR = Path("data/reference_structures")
OUTPUT_DIR = Path("outputs/dl/mixture_of_experts")

# Pattern settings
MIN_ANGLE = 10.0
MAX_ANGLE = 80.0
NUM_POINTS = 1400
WAVELENGTH = "CuKa"
WAVELENGTH_ANGSTROM = 1.5406
REFERENCE_INTENSITY_THRESHOLD = 1.0

# Synthetic-data settings
SYNTH_SAMPLES_PER_FORMULA = 60
RANDOM_SEED = 42
SINGLE_PHASE_FRACTION = 0.15

# Neural-net settings (fixed architecture; no tuning)
# Experts are intentionally lightweight because each one solves a single binary label task.
CNN_CONV_CHANNELS = (6, 12)
CNN_KERNEL_SIZES = (5, 3)
CNN_POOL_KERNEL_SIZE = 2
CNN_ADAPTIVE_POOL_POINTS = 32
NN_HIDDEN_LAYER_SIZES = (16, 8)
NN_ALPHA = 1e-4
NN_LEARNING_RATE_INIT = 1e-3
NN_BATCH_SIZE = 64
NN_MAX_ITER = 80
NN_EARLY_STOPPING_MIN_EPOCHS = 20
NN_EARLY_STOPPING_PATIENCE = 10
NN_EARLY_STOPPING_MIN_DELTA = 1e-4
PREDICTION_THRESHOLD = 0.50
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Artifact ranges
UNIFORM_SHIFT_RANGE = (-0.12, 0.12)
SAMPLE_DISPLACEMENT_RANGE_MM = (-0.18, 0.18)
GONIOMETER_RADIUS_MM = 240.0

U_RANGE = (0.01, 0.06)
V_RANGE = (-0.02, 0.01)
W_RANGE = (0.002, 0.010)
SIZE_NM_RANGE = (8.0, 80.0)
MICROSTRAIN_RANGE = (0.0, 0.003)

FWHM_RANGE = (0.08, 0.45)
ETA_RANGE = (0.10, 0.70)

BACKGROUND_SCALE_RANGE = (0.05, 0.28)
HUMP_SCALE_RANGE = (0.02, 0.20)
NOISE_SCALE_RANGE = (0.002, 0.020)

# Split
VAL_FRACTION = 0.20


# -----------------------------
# Pattern simulation utilities
# -----------------------------
def normalize_0_100(y):
    y = np.asarray(y, dtype=float)
    y = y - y.min()
    return 100.0 * y / np.clip(y.max(), 1e-12, None)


def sample_displacement_shift(two_theta_deg, displacement_mm):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    d_relative_change = displacement_mm / GONIOMETER_RADIUS_MM * np.cos(theta_rad) ** 2
    return np.rad2deg(-d_relative_change * np.tan(theta_rad))


def instrumental_fwhm(two_theta_deg, u, v, w):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    tan_theta = np.tan(theta_rad)
    fwhm_sq = u * tan_theta**2 + v * tan_theta + w
    return np.sqrt(np.clip(fwhm_sq, 1e-4, None))


def size_fwhm(two_theta_deg, size_nm):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    wavelength_nm = WAVELENGTH_ANGSTROM / 10.0
    beta_rad = 0.9 * wavelength_nm / (size_nm * np.cos(theta_rad))
    return np.rad2deg(beta_rad)


def strain_fwhm(two_theta_deg, microstrain):
    theta_rad = np.deg2rad(two_theta_deg / 2.0)
    beta_rad = 4.0 * microstrain * np.tan(theta_rad)
    return np.rad2deg(beta_rad)


def pseudo_voigt_profile(two_theta_grid, centers, fwhm, eta):
    dx = two_theta_grid[:, None] - centers[None, :]
    sigma = np.clip(fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0))), 1e-6, None)
    gamma = np.clip(fwhm / 2.0, 1e-6, None)
    gauss = np.exp(-0.5 * (dx / sigma[None, :]) ** 2)
    lorentz = (gamma[None, :] ** 2) / (dx**2 + gamma[None, :] ** 2)
    return (1.0 - eta) * gauss + eta * lorentz


def load_reference_sticks(cif_files):
    calc = XRDCalculator(wavelength=WAVELENGTH)
    refs_by_formula = {}

    for cif_file in cif_files:
        pattern = calc.get_pattern(Structure.from_file(cif_file), two_theta_range=(MIN_ANGLE, MAX_ANGLE))
        peak_pos = np.asarray(pattern.x, dtype=float)
        peak_int = np.asarray(pattern.y, dtype=float)

        keep = peak_int >= REFERENCE_INTENSITY_THRESHOLD
        peak_pos = peak_pos[keep]
        peak_int = normalize_0_100(peak_int[keep])

        formula = cif_file.stem.split("_", 1)[0]
        refs_by_formula.setdefault(formula, []).append({"phase": cif_file.stem, "peak_pos": peak_pos, "peak_int": peak_int})

    return refs_by_formula


def simulate_component_profile(two_theta_grid, base_pos, base_int, rng):
    if len(base_pos) == 0:
        return np.zeros_like(two_theta_grid)

    peak_int = base_int * np.exp(rng.normal(0.0, 0.25, size=len(base_int)))

    uniform_shift = rng.uniform(*UNIFORM_SHIFT_RANGE)
    displacement = rng.uniform(*SAMPLE_DISPLACEMENT_RANGE_MM)
    peak_pos = base_pos + uniform_shift + sample_displacement_shift(base_pos, displacement)

    keep = (peak_pos >= MIN_ANGLE - 1.0) & (peak_pos <= MAX_ANGLE + 1.0)
    peak_pos = peak_pos[keep]
    peak_int = peak_int[keep]
    if len(peak_pos) == 0:
        return np.zeros_like(two_theta_grid)

    u = rng.uniform(*U_RANGE)
    v = rng.uniform(*V_RANGE)
    w = rng.uniform(*W_RANGE)
    size_nm = rng.uniform(*SIZE_NM_RANGE)
    microstrain = rng.uniform(*MICROSTRAIN_RANGE)

    fwhm = np.sqrt(
        instrumental_fwhm(peak_pos, u, v, w) ** 2
        + size_fwhm(peak_pos, size_nm) ** 2
        + strain_fwhm(peak_pos, microstrain) ** 2
    )
    fwhm += rng.uniform(*FWHM_RANGE)
    eta = rng.uniform(*ETA_RANGE)

    profile = pseudo_voigt_profile(two_theta_grid, peak_pos, fwhm, eta) @ peak_int
    return profile / np.clip(profile.max(), 1e-12, None)


def simulate_multiphase_profile(two_theta_grid, formula_list, refs_by_formula, rng):
    components = []
    for formula in formula_list:
        ref = refs_by_formula[formula][rng.integers(0, len(refs_by_formula[formula]))]
        components.append(simulate_component_profile(two_theta_grid, ref["peak_pos"], ref["peak_int"], rng))

    weights = rng.dirichlet(np.ones(len(components)) * 1.5)
    peaks = np.zeros_like(two_theta_grid)
    for w, comp in zip(weights, components):
        peaks += w * comp
    peaks = normalize_0_100(peaks)

    x_cheb = 2.0 * (two_theta_grid - MIN_ANGLE) / (MAX_ANGLE - MIN_ANGLE) - 1.0
    coeffs = np.array([1.0, rng.uniform(-0.5, 0.5), rng.uniform(-0.4, 0.4), rng.uniform(-0.2, 0.2), rng.uniform(-0.1, 0.1)])
    background = np.polynomial.chebyshev.chebval(x_cheb, coeffs)
    background -= background.min()
    background /= np.clip(background.max(), 1e-12, None)
    background *= rng.uniform(*BACKGROUND_SCALE_RANGE) * peaks.max()

    center = rng.uniform(18.0, 35.0)
    width = rng.uniform(5.0, 12.0)
    hump = rng.uniform(*HUMP_SCALE_RANGE) * peaks.max() * np.exp(-0.5 * ((two_theta_grid - center) / width) ** 2)

    noise_sigma = rng.uniform(*NOISE_SCALE_RANGE) * peaks.max()
    noise = rng.normal(0.0, noise_sigma, size=len(two_theta_grid))

    y = peaks + background + hump + noise
    y -= y.min()
    return normalize_0_100(y)


def build_synthetic_multiphase_dataset(refs_by_formula, two_theta_grid, rng):
    formulas = sorted(refs_by_formula.keys())
    X = []
    y_labels = []

    for anchor in formulas:
        others = [f for f in formulas if f != anchor]
        for _ in range(SYNTH_SAMPLES_PER_FORMULA):
            n_components = int(
                rng.choice(
                    [1, 2, 3],
                    p=[
                        SINGLE_PHASE_FRACTION,
                        (1.0 - SINGLE_PHASE_FRACTION) * 0.65,
                        (1.0 - SINGLE_PHASE_FRACTION) * 0.35,
                    ],
                )
            )
            chosen_others = list(rng.choice(others, size=n_components - 1, replace=False))
            labels = sorted([anchor] + chosen_others)

            profile = simulate_multiphase_profile(two_theta_grid, labels, refs_by_formula, rng)
            X.append(profile)
            y_labels.append(labels)

    return np.asarray(X), y_labels


def preprocess_experimental_pattern(xy_file, two_theta_grid):
    data = np.loadtxt(xy_file)
    x = data[:, 0]
    y = data[:, 1]

    keep = (x >= MIN_ANGLE) & (x <= MAX_ANGLE)
    x = x[keep]
    y = y[keep]

    y_interp = np.interp(two_theta_grid, x, y)
    return normalize_0_100(y_interp)


# -----------------------------
# Mixture-of-experts CNN utils
# -----------------------------
def labels_from_filename(file_stem):
    return file_stem.split("_")


class BinaryConvExpert(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(1, CNN_CONV_CHANNELS[0], kernel_size=CNN_KERNEL_SIZES[0], padding=CNN_KERNEL_SIZES[0] // 2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=CNN_POOL_KERNEL_SIZE),
            nn.Conv1d(
                CNN_CONV_CHANNELS[0],
                CNN_CONV_CHANNELS[1],
                kernel_size=CNN_KERNEL_SIZES[1],
                padding=CNN_KERNEL_SIZES[1] // 2,
            ),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=CNN_POOL_KERNEL_SIZE),
            nn.AdaptiveAvgPool1d(CNN_ADAPTIVE_POOL_POINTS),
        )

        conv_output_points = CNN_ADAPTIVE_POOL_POINTS
        conv_output_size = CNN_CONV_CHANNELS[1] * conv_output_points

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(conv_output_size, NN_HIDDEN_LAYER_SIZES[0]),
            nn.ReLU(),
            nn.Linear(NN_HIDDEN_LAYER_SIZES[0], NN_HIDDEN_LAYER_SIZES[1]),
            nn.ReLU(),
            nn.Linear(NN_HIDDEN_LAYER_SIZES[1], 1),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def build_binary_expert():
    return BinaryConvExpert().to(DEVICE)


def fit_binary_expert(model, X_train_scaled, y_train_bin, X_val_scaled, y_val_bin):
    y_train_bin = y_train_bin.astype(np.float32).reshape(-1, 1)
    y_val_bin = y_val_bin.astype(np.float32).reshape(-1, 1)

    train_dataset = TensorDataset(
        torch.from_numpy(X_train_scaled[:, None, :]),
        torch.from_numpy(y_train_bin),
    )
    train_loader = DataLoader(train_dataset, batch_size=NN_BATCH_SIZE, shuffle=True)
    X_val_tensor = torch.from_numpy(X_val_scaled[:, None, :]).to(DEVICE)
    y_val_tensor = torch.from_numpy(y_val_bin).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=NN_LEARNING_RATE_INIT, weight_decay=NN_ALPHA)

    train_loss_curve = []
    val_loss_curve = []
    best_val_loss = float("inf")
    best_epoch = 0
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    epochs_without_improvement = 0

    for epoch in range(1, NN_MAX_ITER + 1):
        model.train()
        epoch_loss = 0.0
        n_seen = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            batch_size = X_batch.shape[0]
            epoch_loss += loss.item() * batch_size
            n_seen += batch_size

        train_loss = epoch_loss / np.clip(n_seen, 1, None)
        train_loss_curve.append(train_loss)

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_tensor)
            val_loss = criterion(val_logits, y_val_tensor).item()
        val_loss_curve.append(val_loss)

        is_improved = val_loss < (best_val_loss - NN_EARLY_STOPPING_MIN_DELTA)
        if is_improved:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch >= NN_EARLY_STOPPING_MIN_EPOCHS and epochs_without_improvement >= NN_EARLY_STOPPING_PATIENCE:
            break

    model.load_state_dict(best_state)

    return (
        np.asarray(train_loss_curve, dtype=float),
        np.asarray(val_loss_curve, dtype=float),
        int(best_epoch),
    )


def get_binary_scores(model, X_scaled):
    X_tensor = torch.from_numpy(X_scaled[:, None, :]).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(X_tensor).squeeze(1)
        scores = torch.sigmoid(logits).cpu().numpy()

    return scores


def predict_binary_labels(scores, threshold=PREDICTION_THRESHOLD):
    return (scores >= threshold).astype(int)


def compute_micro_metrics(y_true_bin, y_pred_bin):
    precision_micro = precision_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    recall_micro = recall_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    f1_micro = f1_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    return precision_micro, recall_micro, f1_micro


def plot_loss_curve(expert_train_loss_curves, expert_val_loss_curves):
    if len(expert_train_loss_curves) == 0:
        return

    def mean_curve_with_padding(curves_by_formula):
        formulas = sorted(curves_by_formula)
        max_len = max(len(curves_by_formula[f]) for f in formulas)
        matrix = np.full((len(formulas), max_len), np.nan, dtype=float)
        for i, f in enumerate(formulas):
            curve = curves_by_formula[f]
            matrix[i, : len(curve)] = curve
        return np.nanmean(matrix, axis=0), max_len

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for formula in sorted(expert_train_loss_curves):
        train_curve = expert_train_loss_curves[formula]
        val_curve = expert_val_loss_curves[formula]
        ax.plot(np.arange(1, len(train_curve) + 1), train_curve, color="tab:blue", alpha=0.12, linewidth=0.9)
        ax.plot(np.arange(1, len(val_curve) + 1), val_curve, color="tab:orange", alpha=0.10, linewidth=0.9)

    train_mean, train_len = mean_curve_with_padding(expert_train_loss_curves)
    val_mean, val_len = mean_curve_with_padding(expert_val_loss_curves)
    ax.plot(np.arange(1, train_len + 1), train_mean, color="tab:blue", linewidth=2.3, label="Train (mean expert)")
    ax.plot(np.arange(1, val_len + 1), val_mean, color="tab:orange", linewidth=2.1, label="Val (mean expert)")

    ax.set_xlabel("Epoch", fontsize=18, labelpad=8)
    ax.set_ylabel("Loss", fontsize=18, labelpad=10)
    ax.tick_params(axis="both", labelsize=16)
    ax.legend(fontsize=16, loc="upper right", framealpha=1)
    ax.grid(alpha=0.25)

    out_file = OUTPUT_DIR / "nn_loss_curve.png"
    plt.tight_layout()
    plt.savefig(out_file, dpi=200)
    plt.close(fig)
    print(f"Saved plot: {out_file}")


def plot_test_metric_summary(precision_micro, recall_micro, f1_micro):
    fig, ax = plt.subplots(figsize=(6.6, 4.5))

    x = np.array([0])
    width = 0.24
    ax.bar(x - width, [precision_micro], width=width, color="tab:blue", edgecolor="black", linewidth=1.0, label="Precision")
    ax.bar(x, [recall_micro], width=width, color="tab:green", edgecolor="black", linewidth=1.0, label="Recall")
    ax.bar(x + width, [f1_micro], width=width, color="tab:red", edgecolor="black", linewidth=1.0, label="F1-score")

    ax.set_xticks(x)
    ax.set_xticklabels(["MoE-CNN"], fontsize=16)
    ax.tick_params(axis="y", labelsize=16)
    ax.set_ylim(0.0, 1.05)
    ax.set_ylabel("Score", fontsize=18, labelpad=12)
    ax.legend(fontsize=16, loc="lower right", framealpha=1)
    ax.grid(axis="y", alpha=0.25)

    out_file = OUTPUT_DIR / "nn_test-metric_summary.png"
    plt.tight_layout()
    plt.savefig(out_file, dpi=200)
    plt.close(fig)
    print(f"Saved plot: {out_file}")


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(RANDOM_SEED)
    two_theta_grid = np.linspace(MIN_ANGLE, MAX_ANGLE, NUM_POINTS)

    refs_by_formula = load_reference_sticks(sorted(REFERENCE_DIR.glob("*.cif")))
    all_formulas = sorted(refs_by_formula.keys())

    X, y_label_lists = build_synthetic_multiphase_dataset(refs_by_formula, two_theta_grid, rng)

    mlb = MultiLabelBinarizer(classes=all_formulas)
    y_bin = mlb.fit_transform(y_label_lists)

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y_bin,
        test_size=VAL_FRACTION,
        random_state=RANDOM_SEED,
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
    X_val_scaled = scaler.transform(X_val).astype(np.float32)

    exp_files = sorted(EXPERIMENT_DIR.glob("*.xy"))
    X_test = np.asarray([preprocess_experimental_pattern(f, two_theta_grid) for f in exp_files])
    X_test_scaled = scaler.transform(X_test).astype(np.float32)
    y_test_labels = [labels_from_filename(f.stem) for f in exp_files]
    y_test = mlb.transform(y_test_labels)

    print("\n=== CNN Mixture-of-Experts Demo (multi-phase) ===")
    print(f"Synthetic samples: {len(X)}")
    print(f"Unique formulas (labels): {len(all_formulas)}")
    print(f"Training samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    print(f"Experimental test patterns: {len(X_test)}")

    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)

    expert_models = {}
    expert_train_loss_curves = {}
    expert_val_loss_curves = {}
    val_scores_by_formula = []
    test_scores_by_formula = []
    expert_metric_rows = []
    expert_epochs = []

    print("\nExperts")
    for i, formula in enumerate(all_formulas):
        torch.manual_seed(RANDOM_SEED + i)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(RANDOM_SEED + i)

        model = build_binary_expert()
        train_loss_curve, val_loss_curve, best_epoch = fit_binary_expert(
            model,
            X_train_scaled,
            y_train[:, i],
            X_val_scaled,
            y_val[:, i],
        )
        epochs_ran = len(train_loss_curve)

        val_scores = get_binary_scores(model, X_val_scaled)
        val_preds = predict_binary_labels(val_scores, threshold=PREDICTION_THRESHOLD)

        test_scores = get_binary_scores(model, X_test_scaled)

        val_precision = precision_score(y_val[:, i], val_preds, zero_division=0)
        val_recall = recall_score(y_val[:, i], val_preds, zero_division=0)
        val_f1 = f1_score(y_val[:, i], val_preds, zero_division=0)

        print(
            f"  {formula:10s} -> output neuron in [0, 1], threshold={PREDICTION_THRESHOLD:.2f}, "
            f"epochs={epochs_ran}, best_epoch={best_epoch}, val F1={val_f1:.3f}"
        )

        expert_models[formula] = model
        expert_train_loss_curves[formula] = train_loss_curve
        expert_val_loss_curves[formula] = val_loss_curve
        val_scores_by_formula.append(val_scores)
        test_scores_by_formula.append(test_scores)
        expert_epochs.append(epochs_ran)
        expert_metric_rows.append(
            {
                "formula": formula,
                "threshold": PREDICTION_THRESHOLD,
                "epochs_ran": epochs_ran,
                "best_epoch": best_epoch,
                "validation_precision": val_precision,
                "validation_recall": val_recall,
                "validation_f1": val_f1,
            }
        )

    val_score_matrix = np.column_stack(val_scores_by_formula)
    test_score_matrix = np.column_stack(test_scores_by_formula)

    y_pred_val = predict_binary_labels(val_score_matrix, threshold=PREDICTION_THRESHOLD)
    y_pred_test = predict_binary_labels(test_score_matrix, threshold=PREDICTION_THRESHOLD)

    val_precision_micro, val_recall_micro, val_f1_micro = compute_micro_metrics(y_val, y_pred_val)
    test_precision_micro, test_recall_micro, test_f1_micro = compute_micro_metrics(y_test, y_pred_test)

    print("\nMixture of Experts")
    print(f"  Conv layers per expert: {CNN_CONV_CHANNELS} kernels={CNN_KERNEL_SIZES} pool={CNN_POOL_KERNEL_SIZE}")
    print(f"  Hidden layers per expert: {NN_HIDDEN_LAYER_SIZES}")
    print(f"  Experts (one per label): {len(all_formulas)}")
    print("  Output neuron per expert: 1 (sigmoid score in [0, 1])")
    print(f"  Binary threshold: {PREDICTION_THRESHOLD:.2f}")
    print(f"  Max epochs per expert: {NN_MAX_ITER}")
    print(
        "  Early stopping: "
        f"min_epochs={NN_EARLY_STOPPING_MIN_EPOCHS}, "
        f"patience={NN_EARLY_STOPPING_PATIENCE}, "
        f"min_delta={NN_EARLY_STOPPING_MIN_DELTA}"
    )
    print(f"  Mean epochs actually run: {np.mean(expert_epochs):.1f}")
    print(f"  Validation precision (micro): {val_precision_micro:.3f}")
    print(f"  Validation recall (micro): {val_recall_micro:.3f}")
    print(f"  Validation F1 (micro): {val_f1_micro:.3f}")
    print(f"  Test precision (micro): {test_precision_micro:.3f}")
    print(f"  Test recall (micro): {test_recall_micro:.3f}")
    print(f"  Test F1 (micro): {test_f1_micro:.3f}")

    print("  Test predictions (binary outputs):")
    prediction_rows = []
    for i, exp_file in enumerate(exp_files):
        true_set = y_test_labels[i]
        pred_set = list(mlb.classes_[np.where(y_pred_test[i] == 1)[0]])

        true_binary = " ".join(map(str, y_test[i].astype(int).tolist()))
        pred_binary = " ".join(map(str, y_pred_test[i].astype(int).tolist()))
        inter = len(set(true_set) & set(pred_set))
        f1_pattern = 0.0 if (len(true_set) + len(pred_set)) == 0 else 2.0 * inter / (len(true_set) + len(pred_set))

        print(f"    {exp_file.stem:26s} -> [{pred_binary}] {pred_set}")
        prediction_rows.append(
            {
                "pattern": exp_file.stem,
                "true_labels": ";".join(true_set),
                "predicted_labels": ";".join(pred_set),
                "true_binary": true_binary,
                "predicted_binary": pred_binary,
                "n_predicted": len(pred_set),
                "f1_pattern": f1_pattern,
            }
        )

    plot_loss_curve(expert_train_loss_curves, expert_val_loss_curves)
    plot_test_metric_summary(test_precision_micro, test_recall_micro, test_f1_micro)

    metrics_file = OUTPUT_DIR / "nn_metrics.csv"
    with open(metrics_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "model",
                "hidden_layers",
                "n_experts",
                "threshold",
                "max_iter",
                "early_stopping_min_epochs",
                "early_stopping_patience",
                "early_stopping_min_delta",
                "mean_epochs_ran",
                "validation_precision_micro",
                "validation_recall_micro",
                "validation_f1_micro",
                "test_precision_micro",
                "test_recall_micro",
                "test_f1_micro",
            ],
        )
        writer.writeheader()
        writer.writerow(
            {
                "model": "CNN-MoE",
                "hidden_layers": str(NN_HIDDEN_LAYER_SIZES),
                "n_experts": len(all_formulas),
                "threshold": PREDICTION_THRESHOLD,
                "max_iter": NN_MAX_ITER,
                "early_stopping_min_epochs": NN_EARLY_STOPPING_MIN_EPOCHS,
                "early_stopping_patience": NN_EARLY_STOPPING_PATIENCE,
                "early_stopping_min_delta": NN_EARLY_STOPPING_MIN_DELTA,
                "mean_epochs_ran": float(np.mean(expert_epochs)),
                "validation_precision_micro": val_precision_micro,
                "validation_recall_micro": val_recall_micro,
                "validation_f1_micro": val_f1_micro,
                "test_precision_micro": test_precision_micro,
                "test_recall_micro": test_recall_micro,
                "test_f1_micro": test_f1_micro,
            }
        )

    pred_file = OUTPUT_DIR / "test_predictions_binary.csv"
    with open(pred_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "pattern",
                "true_labels",
                "predicted_labels",
                "true_binary",
                "predicted_binary",
                "n_predicted",
                "f1_pattern",
            ],
        )
        writer.writeheader()
        writer.writerows(prediction_rows)

    expert_metrics_file = OUTPUT_DIR / "expert_metrics.csv"
    with open(expert_metrics_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "formula",
                "threshold",
                "epochs_ran",
                "best_epoch",
                "validation_precision",
                "validation_recall",
                "validation_f1",
            ],
        )
        writer.writeheader()
        writer.writerows(expert_metric_rows)

    print(f"\nSaved metrics: {metrics_file}")
    print(f"Saved predictions: {pred_file}")
    print(f"Saved expert metrics: {expert_metrics_file}")


# 04f — Mixture of Experts

One lightweight binary expert is trained per phase label, then combined for final multi-label prediction.

## Option A / Option B
Option A: run a short MoE training demo.

Option B: load per-expert checkpoints from `data/pretrained/` in a custom workflow.

In [ ]:
PRETRAINED_PATH = "data/pretrained/moe_experts"
print("Found pretrained expert folder:", os.path.exists(PRETRAINED_PATH))

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
SYNTH_SAMPLES_PER_FORMULA = 6
NN_MAX_ITER = 20  # Reduced for demo speed
NN_EARLY_STOPPING_MIN_EPOCHS = 5
NN_EARLY_STOPPING_PATIENCE = 4
main()


## What To Observe
MoE can improve flexibility by allowing each phase detector to specialize.

In [ ]:
display(Image("outputs/dl/mixture_of_experts/nn_loss_curve.png"))
display(Image("outputs/dl/mixture_of_experts/nn_test-metric_summary.png"))

## Summary
- MoE decomposes multi-label classification into expert subproblems.
- Early stopping helps keep many experts computationally manageable.
- Expert-level metrics can reveal label-specific weaknesses.

## Next Steps
Continue to **04 — Open Challenge**: [Open in Colab](https://colab.research.google.com/github/Szymanski-Group/MRS_CH08_Tutorial/blob/main/notebooks/04_Challenge.ipynb)